# RAW DATA EDA V2 — Data quality profiling & Silver cleaning specification

Notebook này khám phá Raw độc lập, tạo bằng chứng và đề xuất để review trước khi viết Silver. Không gọi pipeline, Spark, Iceberg, MinIO, Trino, Airflow hoặc đọc kết quả Silver. Các bảng phân tích là projection trong bộ nhớ; Raw luôn giữ nguyên.

**Cách dùng:** Run All từ kernel sạch. STEP 0 chọn source/file, đường dẫn qua biến môi trường nếu chạy ngoài cây dự án. STEP 1–9 chứa số liệu được tính lại. Hàm drill_down() xem một file; bảng đầy đủ được giữ trong biến, màn hình giới hạn số dòng. Cuối notebook có exporter Markdown **tắt mặc định**, không ghi đè tài liệu.

**Đếm rõ ràng:** CSV gồm mọi record được parser đọc, kể cả record rỗng có dấu phân cách. Excel phân biệt grid vật lý, sheet metadata và observation view. Counts không dùng sampling; dòng CSV là số record (không phải dòng vật lý nếu có multiline). Cells thiếu cấu trúc, tokens và số 0 được báo riêng. Quy tắc VERIFIED có bằng chứng độc lập, không đồng nghĩa đã được phê duyệt triển khai.


## STEP 0 — Configuration

Root được tìm từ EDA_PROJECT_ROOT, EDA_RAW_ROOT, vị trí notebook (__file__ nếu có), hoặc cây tổ tiên của working directory. Nếu kernel nằm hoàn toàn ngoài dự án, đặt biến môi trường; không có đường dẫn Windows hardcode. SOURCE_FILTER/FILE_FILTER chỉ giới hạn dữ liệu phân tích. Khi chọn NSO, mọi V06.* vẫn được đọc để phân nhóm đúng.

Không thay NULL bằng token hoặc ngược lại lúc đọc. CSV đọc lexical strings; Excel giữ cell types và blank values bằng openpyxl rồi đưa vào pandas. Tệp lớn đọc chunk, hợp nhất categorical dictionaries để kiểm tra duplicate trên toàn bộ dữ liệu, không dùng hash thay cho equality.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import date
import csv, gc, hashlib, io, json, os, re, unicodedata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

SOURCE_NAMES = ("faostat", "nso", "thitruongnongsan", "usda", "world_bank")
SOURCE_FILTER = None           # e.g. {"nso"} or {"faostat", "usda"}
FILE_FILTER = None             # glob relative to raw, e.g. "faostat/price.csv"
CHUNK_ROWS = 100_000
LARGE_CSV_BYTES = 32 * 1024**2
MAX_INPUT_BYTES = 1024 * 1024**2
DISPLAY_ROWS = 18
WRITE_SPECS = False            # explicit opt-in; existing Markdown never overwritten
AS_OF_DATE = pd.Timestamp("2026-10-11")  # reproducible review date; update deliberately
MISSING_TOKEN_CANDIDATES = ("..", "NUL", "NULL", "NA", "N/A", "-", "...", "…")
pd.set_option("display.max_rows", DISPLAY_ROWS)
pd.set_option("display.max_columns", 16)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.width", 160)

def find_raw_root():
    hints = [os.environ.get("EDA_RAW_ROOT"), os.environ.get("EDA_PROJECT_ROOT")]
    if "__file__" in globals():
        hints.append(str(Path(__file__).resolve().parent))
    hints.append(str(Path.cwd()))
    for hint in filter(None, hints):
        start = Path(hint).expanduser().resolve()
        for base in (start, *start.parents):
            for candidate in (base, base / "raw", base / "data_main" / "raw"):
                if all((candidate / s).is_dir() for s in SOURCE_NAMES):
                    return candidate.resolve()
    raise FileNotFoundError("Set EDA_RAW_ROOT or EDA_PROJECT_ROOT to the project containing all five raw source folders.")

RAW_ROOT = find_raw_root()
PROJECT_ROOT = RAW_ROOT.parent.parent if RAW_ROOT.parent.name == "data_main" else RAW_ROOT.parent
selected_sources = set(SOURCE_NAMES if SOURCE_FILTER is None else SOURCE_FILTER)
assert selected_sources <= set(SOURCE_NAMES), "Unknown source"
def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024**2), b""):
            h.update(block)
    return h.hexdigest()
def show(table, n=DISPLAY_ROWS):
    display(table.head(n))
    if len(table) > n:
        print(f"Displayed {n}/{len(table)} rows; full table remains in memory.")
def stable_hash(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:16]
def text_series(s):
    return s.astype("string")
def candidate_name(value):
    s = unicodedata.normalize("NFKD", str(value)).replace("đ", "d").replace("Đ", "D")
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
def as_text(v):
    return "" if v is None or pd.isna(v) else str(v)
protected_files = sorted(p for s in SOURCE_NAMES for p in (RAW_ROOT / s).rglob("*")
                         if p.is_file() and not p.name.startswith("cleaning_"))
protected_files += sorted(p for p in RAW_ROOT.glob("*.ipynb") if p.name != "EDA_v2.ipynb")
initial_hashes = {p.relative_to(RAW_ROOT).as_posix(): sha256_file(p) for p in protected_files}
print("Root located; analysis date:", AS_OF_DATE.date(), "| Sources:", sorted(selected_sources))


Root located; analysis date: 2026-10-11 | Sources: ['faostat', 'nso', 'thitruongnongsan', 'usda', 'world_bank']


## STEP 1 — Data Inventory

Phân loại observation files, bảng mã tham chiếu, metadata và supporting documents. Workbook đọc mọi sheet, kể cả sheet trống. USDA .xls chọn reader theo magic bytes. Header được giữ nguyên theo vị trí; CSV không dùng pandas tự thêm suffix cho header trùng.

schema_fingerprint chỉ dùng thứ tự header + layout (không dùng row count/dtype token bị nhiễu). content_sha256 xác định snapshot; dtypes và coverage có fingerprint riêng. Trong Excel, row count vật lý là max_row của worksheet; observation count được tính riêng ở STEP 2.


In [2]:
def classify(path):
    suffix = path.suffix.lower()
    if suffix in {".md", ".ipynb", ".py"}:
        return "support_document"
    if suffix == ".txt":
        return "source_metadata"
    if suffix in {".csv", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"}:
        return "reference_data" if re.search(r"_(Flags|Elements|ItemCodes|PartnerCountries|ReporterCountries)\.csv$", path.name) else "observation_data"
    return "other_support"

discovered = sorted(p for s in SOURCE_NAMES for p in (RAW_ROOT/s).rglob("*") if p.is_file())
file_inventory = pd.DataFrame([{"source": p.relative_to(RAW_ROOT).parts[0],
    "file": p.relative_to(RAW_ROOT).as_posix(), "extension": p.suffix.lower(),
    "role": classify(p), "bytes": p.stat().st_size} for p in discovered])
show(file_inventory)
DATA = {}
inventory_rows = []

def csv_encoding(path):
    # Strict UTF-8 first. Single-byte fallback is a decoder candidate, not proof of original encoding.
    try:
        with path.open(encoding="utf-8-sig", errors="strict") as stream:
            while stream.read(1024**2):
                pass
        return "utf-8-sig", "strict UTF-8 decode succeeded"
    except UnicodeDecodeError:
        try:
            path.read_bytes().decode("cp1252", errors="strict")
            return "cp1252", "UTF-8 failed; cp1252 decoder candidate; original encoding unconfirmed"
        except UnicodeDecodeError:
            path.read_bytes().decode("latin-1")
            return "latin-1", "lossless byte mapping only; original encoding unconfirmed"

def register(path, table, frame, headers, layout, encoding, evidence, header_rows, physical_rows, cell_types=None, sheet_role="observations"):
    rel = path.relative_to(RAW_ROOT).as_posix()
    ident = rel + ("::" + table if table else "")
    meta = dict(id=ident, source=rel.split("/")[0], file=rel, table=table,
        format=layout, encoding=encoding, encoding_evidence=evidence,
        bytes=path.stat().st_size, rows=len(frame), columns=frame.shape[1],
        physical_rows=physical_rows, header_rows=header_rows, headers=list(headers),
        schema_fingerprint=stable_hash({"headers":list(headers),"layout":layout}),
        content_sha256=initial_hashes[rel], scope="FULL_DATA", role=sheet_role)
    DATA[ident] = {"raw":frame, "meta":meta, "cell_types":cell_types}
    inventory_rows.append({k:v for k,v in meta.items() if k not in {"headers"}})

def read_csv_lossless(path):
    encoding, evidence = csv_encoding(path)
    with path.open(encoding=encoding, newline="") as stream:
        headers = next(csv.reader(stream))
    options = dict(encoding=encoding, header=None, skiprows=1, dtype=str, keep_default_na=False,
                   na_filter=False, skip_blank_lines=False, on_bad_lines="error", low_memory=False)
    if path.stat().st_size > MAX_INPUT_BYTES:
        raise MemoryError(f"{path.name}: input exceeds configured safety limit; raise limit after memory review or implement streaming.")
    if path.stat().st_size >= LARGE_CSV_BYTES:
        chunks = [chunk.astype("category") for chunk in pd.read_csv(path, chunksize=CHUNK_ROWS, **options)]
        frame = pd.DataFrame({c:pd.api.types.union_categoricals([chunk[c].array for chunk in chunks], sort_categories=True)
                              for c in chunks[0].columns})
        del chunks
        gc.collect()
    else:
        frame = pd.read_csv(path, **options)
    if frame.shape[1] != len(headers):
        raise ValueError(f"{path}: header/body column mismatch")
    frame.columns = headers
    register(path, "", frame, headers, "csv", encoding, evidence, [1], len(frame)+1,
             sheet_role="reference_data" if classify(path)=="reference_data" else "observations")

def read_workbook(path):
    import openpyxl
    book = openpyxl.load_workbook(path, read_only=True, data_only=False)
    try:
        for sheet in book:
            if sheet.max_row == 1 and sheet.max_column == 1 and sheet.cell(1,1).value is None:
                grid = pd.DataFrame()
                types = pd.DataFrame()
            else:
                values, cell_types = [], []
                for row in sheet.iter_rows():
                    values.append([cell.value for cell in row])
                    cell_types.append([cell.data_type for cell in row])
                grid = pd.DataFrame(values, dtype=object)
                types = pd.DataFrame(cell_types)
            # Only a direct table header at physical row 1 is a Raw field header.
            direct = bool(len(grid) and all(isinstance(v,str) and v.strip() for v in grid.iloc[0]))
            if direct:
                headers = grid.iloc[0].tolist()
                frame = grid.iloc[1:].copy()
                frame.columns = headers
                types = types.iloc[1:].copy()
                types.columns = headers
                header_rows = [1]
            else:
                headers = list(grid.columns)  # positional grid labels; source header rows remain inside raw
                frame = grid
                header_rows = []
            role = "metadata" if sheet.title in {"Description","Index Weights","AFOSHEET"} else "observations"
            register(path, sheet.title, frame, headers, "xlsx", "OOXML Unicode",
                     "openpyxl cell values/types, formulas retained", header_rows, sheet.max_row, types, role)
    finally:
        book.close()

def read_html_xls(path):
    from lxml import html
    text = path.read_text(encoding="utf-8-sig")
    doc = html.fromstring(text)
    for index, table in enumerate(doc.xpath("//table")):
        rows = table.xpath(".//tr")
        headers = ["".join(c.itertext()).strip() for c in rows[0].xpath("./th|./td")]
        values = [["".join(c.itertext()).strip() for c in row.xpath("./th|./td")] for row in rows[1:]]
        frame = pd.DataFrame(values, columns=headers)
        register(path, f"html_table_{index+1}", frame, headers, "html (.xls extension)",
                 "utf-8", "HTML meta charset=utf-8; lexical cells preserved", [1],len(rows))

selected_paths = []
for p in discovered:
    source = p.relative_to(RAW_ROOT).parts[0]
    if source not in selected_sources or classify(p) not in {"observation_data","reference_data"}:
        continue
    rel = p.relative_to(RAW_ROOT)
    if FILE_FILTER and not rel.match(FILE_FILTER):
        if not (source == "nso" and p.match("V06.*.csv")):
            continue
    selected_paths.append(p)
for p in selected_paths:
    if p.suffix.lower()==".csv":
        read_csv_lossless(p)
    elif p.suffix.lower()==".xlsx":
        read_workbook(p)
    elif p.suffix.lower()==".xls" and p.read_bytes()[:512].lstrip().lower().startswith(b"<html"):
        read_html_xls(p)
    elif p.suffix.lower()==".xls":
        # Genuine binary XLS needs an installed pandas engine; failures remain explicit.
        for sheet, frame in pd.read_excel(p, sheet_name=None, dtype=object, keep_default_na=False).items():
            register(p,sheet,frame,list(frame.columns),"binary xls","binary","pandas XLS engine",[1],len(frame)+1)
    elif p.suffix.lower()==".parquet":
        frame=pd.read_parquet(p); register(p,"",frame,list(frame.columns),"parquet","binary","pandas parquet",[1],len(frame))
    elif p.suffix.lower() in {".json",".jsonl"}:
        frame=pd.read_json(p,lines=p.suffix.lower()==".jsonl")
        register(p,"",frame,list(frame.columns),p.suffix[1:],"utf-8","pandas JSON",[],len(frame))
assert DATA, "No selected data files"
inventory = pd.DataFrame(inventory_rows)
print(f"Loaded {len(selected_paths)} files / {len(DATA)} tables, full-data profiling.")
show(inventory[["source","file","table","role","format","encoding","bytes","rows","columns","physical_rows","schema_fingerprint"]],50)

def drill_down(source=None, file=None, rows=5):
    matches = [(ident,entry) for ident,entry in DATA.items()
               if (source is None or entry["meta"]["source"]==source)
               and (file is None or file in ident)]
    for ident,entry in matches:
        display(Markdown("**"+ident+"**"))
        display(pd.DataFrame([entry["meta"]]))
        display(entry["raw"].head(rows))
    return [ident for ident,_ in matches]


,source,file,extension,role,bytes
0,faostat,faostat/eda_faostat.md,.md,support_document,19599
1,faostat,faostat/price.csv,.csv,observation_data,2509272
2,faostat,faostat/production_world.csv,.csv,observation_data,3292244
3,faostat,faostat/supply_utilization.csv,.csv,observation_data,6096569
4,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,.csv,observation_data,167990400
5,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Eleme...,.csv,reference_data,236
6,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Flags...,.csv,reference_data,133
7,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_ItemC...,.csv,reference_data,21739
8,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Partn...,.csv,reference_data,5769
9,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Repor...,.csv,reference_data,5161


Displayed 18/32 rows; full table remains in memory.


Loaded 26 files / 30 tables, full-data profiling.


,source,file,table,role,format,encoding,bytes,rows,columns,physical_rows,schema_fingerprint
0,faostat,faostat/price.csv,,observations,csv,utf-8-sig,2509272,15334,16,15335,1e2e1032e9a48100
1,faostat,faostat/production_world.csv,,observations,csv,utf-8-sig,3292244,22738,15,22739,d1e64ff0245aedbe
2,faostat,faostat/supply_utilization.csv,,observations,csv,utf-8-sig,6096569,36380,15,36381,d1e64ff0245aedbe
3,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,,observations,csv,utf-8-sig,167990400,1226470,16,1226471,8fb828a43896c0df
4,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Eleme...,,reference_data,csv,utf-8-sig,236,10,2,11,6c6ca62af7a54514
...,...,...,...,...,...,...,...,...,...,...,...
25,world_bank,world_bank/CMO-Historical-Data-Monthly.xlsx,AFOSHEET,metadata,xlsx,OOXML Unicode,778415,0,0,1,93e9c2361c34788b
26,world_bank,world_bank/CMO-Historical-Data-Monthly.xlsx,Monthly Prices,observations,xlsx,OOXML Unicode,778415,798,89,798,865d4a9ce34b74ca
27,world_bank,world_bank/CMO-Historical-Data-Monthly.xlsx,Monthly Indices,observations,xlsx,OOXML Unicode,778415,1362,81,1362,9ace360185409737
28,world_bank,world_bank/CMO-Historical-Data-Monthly.xlsx,Description,metadata,xlsx,OOXML Unicode,778415,403,87,403,f96adad8b96bb19a


## STEP 2 — Schema Profiling & observation projections

Bảng schema dùng toàn bộ cột, báo inferred dtype, mixed types, header trùng/rỗng, normalized-name collisions và dấu hiệu encoding. Normalized names chỉ là candidates. Header CSV và raw Excel grids không bị sửa.

NSO phân nhóm bằng fingerprint + data organization, không theo số dòng hoặc tên file. Geography coverage khác nhau được ghi trong cùng nhóm. Source metadata.txt cho tên chỉ tiêu/mùa vụ; đơn vị thiếu vẫn chờ xác nhận. Các observation views unpivot chỉ phục vụ EDA, giữ raw_row/raw_column/raw_value và không xuất dữ liệu đã làm sạch.


In [3]:
def numeric_probe(s):
    t = text_series(s).str.strip()
    thousand = t.str.fullmatch(r"[+-]?\d{1,3}(?:,\d{3})+(?:\.\d+)?", na=False)
    candidate = t.mask(thousand, t.str.replace(",", "", regex=False))
    return pd.to_numeric(candidate, errors="coerce"), thousand

def year_header(c):
    return bool(re.fullmatch(r"\d{4}",str(c)) or
                re.fullmatch(r"\d{4}/\d{4}",str(c)) or
                (re.search(r"(19|20)\d{2}",str(c)) and str(c).lower().startswith("so b")))

def structural_mask(d):
    if d.shape[1]==0:
        return pd.Series(False,index=d.index,dtype=bool)
    return d.apply(lambda s: text_series(s).fillna("").str.strip().eq("")).all(axis=1)

schema_rows = []
for ident,entry in DATA.items():
    d=entry["raw"]; headers=entry["meta"]["headers"]
    normalized=[candidate_name(c) for c in headers]
    for pos,c in enumerate(headers):
        s=d.iloc[:,pos]; t=text_series(s); numeric,_=numeric_probe(s)
        present=~(s.isna() | t.fillna("").str.strip().eq("") | t.str.strip().isin(MISSING_TOKEN_CANDIDATES))
        good=present & numeric.notna()
        wrong=present & numeric.isna()
        unique=t.dropna().unique().tolist()
        leading=t.str.match(r"^'?0\d+",na=False)
        encoding=t.str.contains(r"\?|�|Ã.|Â.|â€",regex=True,na=False)
        semantic=candidate_name(c)
        roles=[]
        if re.search(r"year|nam|date|ngay|period|month",semantic) or year_header(c): roles.append("time")
        if re.search(r"area|country|reporter|partner|market|thi_truong|t_nh",semantic): roles.append("geography")
        if re.search(r"unit|don_vi|element|attribute|statistic|item|commodity|mat_hang",semantic): roles.append("measure/unit")
        if re.search(r"flag|note|domain|table|nguon|source",semantic): roles.append("provenance")
        inferred = "numeric" if int(good.sum()) and not int(wrong.sum()) else ("mixed numeric/text" if int(good.sum()) else "text/empty")
        native_types=s.dropna().map(lambda v:type(v).__name__).value_counts().to_dict() if entry["meta"]["format"]=="xlsx" else {"lexical_str":len(s)}
        schema_rows.append(dict(id=ident,column_position=pos+1,raw_column=str(c),normalized_candidate=normalized[pos],
            storage_dtype=str(s.dtype),inferred_dtype=inferred,native_types=str(native_types),
            numeric_count=int(good.sum()),non_numeric_count=int(wrong.sum()),
            distinct_count=s.nunique(dropna=False),null_count=int(s.isna().sum()),
            duplicate_header=headers.count(c)>1,blank_header=c is None or not str(c).strip() or str(c).startswith("Unnamed"),
            normalized_collision=normalized.count(normalized[pos])>1,
            leading_zero_count=int(leading.sum()),encoding_suspect_count=int(encoding.sum()),
            header_encoding_suspect=bool(re.search(r"\?|�|Ã.|Â.|â€",str(c))),
            field_role=", ".join(roles), examples=str(unique[:4])))
schema_profile=pd.DataFrame(schema_rows)
show(schema_profile)
show(schema_profile.loc[schema_profile.duplicate_header | schema_profile.blank_header |
                       schema_profile.leading_zero_count.gt(0) | schema_profile.header_encoding_suspect |
                       schema_profile.inferred_dtype.eq("mixed numeric/text")],35)

# Full NSO grouping; only data organization and literal header fingerprint define processing groups.
nso_members=[]
for ident,entry in DATA.items():
    if entry["meta"]["source"]!="nso":continue
    d=entry["raw"]; years=[c for c in d.columns if year_header(c)]
    organization="geography_year_matrix" if years else ("statistic_year_measure_table" if "Nam" in d.columns else "unrecognized")
    entry["organization"]=organization
    coverage=sorted(text_series(d.iloc[:,0]).dropna().str.strip().loc[lambda x:x.ne("")].unique().tolist())
    nso_members.append(dict(id=ident,schema_fingerprint=entry["meta"]["schema_fingerprint"],organization=organization,
        raw_rows=len(d),nonstructural_rows=int((~structural_mask(d)).sum()),year_fields=len(years),
        year_span=(str(years[0])+" → "+str(years[-1])) if years else "Nam values",
        coverage_fingerprint=stable_hash(coverage),coverage_labels=len(coverage)))
nso_groups=pd.DataFrame(nso_members)
if len(nso_groups):
    signatures=sorted(set(zip(nso_groups.organization,nso_groups.schema_fingerprint)), key=lambda x:(x[0]!="statistic_year_measure_table",x))
    group_ids={sig:f"schema_group_{i+1:02d}" for i,sig in enumerate(signatures)}
    nso_groups["group"]= [group_ids[(r.organization,r.schema_fingerprint)] for r in nso_groups.itertuples()]
    # Assign national first for readable file numbering; order determined by organization, not file names.
    show(nso_groups,50)

# Views are additional analysis objects; raw frames never renamed/filled/dropped.
VIEWS={}
WB_CONTEXT={}
for ident,entry in DATA.items():
    d=entry["raw"]; m=entry["meta"]; source=m["source"]
    if source=="nso":
        active=d.loc[~structural_mask(d)]
        if entry["organization"]=="geography_year_matrix":
            years=[c for c in d.columns if year_header(c)]
            tagged=active.copy(); tagged["raw_row"]=tagged.index+2
            view=tagged.melt(id_vars=[d.columns[0],"raw_row"],value_vars=years,var_name="raw_column",value_name="raw_value")
            view=view.rename(columns={d.columns[0]:"geography_raw"})
            view["period_raw"]=view.raw_column
            view["statistic_kind"]="physical_quantity"
        elif entry["organization"]=="statistic_year_measure_table":
            measures=[c for c in d.columns if c not in d.columns[:2]]
            tagged=active.copy(); tagged["raw_row"]=tagged.index+2
            view=tagged.melt(id_vars=[d.columns[0],d.columns[1],"raw_row"],value_vars=measures,
                var_name="raw_column",value_name="raw_value").rename(columns={d.columns[0]:"statistic_raw",d.columns[1]:"period_raw"})
            view["statistic_kind"]=np.where(view.statistic_raw.astype(str).str.contains("100"),"development_index","physical_quantity")
            view["geography_raw"]="national (table scope)"
        else:
            raise ValueError(f"Unrecognized NSO layout: {ident}")
        view["year_candidate"]=pd.to_numeric(view.period_raw.astype(str).str.extract(r"((?:19|20)\d{2})",expand=False),errors="coerce")
        view["provisional"]=view.period_raw.astype(str).str.lower().str.startswith("so b")
        VIEWS[ident]=view
    elif source=="usda" and m["format"].startswith("html"):
        periods=[c for c in d.columns if re.fullmatch(r"\d{4}/\d{4}",str(c))]
        temp=d.copy();temp["raw_row"]=temp.index+2
        view=temp.melt(id_vars=["Commodity","Attribute","Country","Unit Description","raw_row"],
            value_vars=periods,var_name="period_raw",value_name="raw_value")
        view["raw_column"]=view.period_raw
        VIEWS[ident]=view
    elif source=="world_bank" and m["table"] in {"Monthly Prices","Monthly Indices"}:
        first=text_series(d.iloc[:,0])
        date_rows=first.str.fullmatch(r"\d{4}M\d{2}",na=False)
        if not date_rows.any():
            raise ValueError(f"No monthly period grid: {ident}")
        start=int(d.index[date_rows][0])
        names={}
        units={}
        for col in d.columns[1:]:
            if m["table"]=="Monthly Prices":
                names[col]=as_text(d.loc[start-2,col])
                units[col]=as_text(d.loc[start-1,col])
            else:
                labels=[as_text(v).strip() for v in d.loc[4:start-1,col] if as_text(v).strip()]
                names[col]=" / ".join(labels)
                units[col]="index (2010=100)"
        WB_CONTEXT[ident]={"header_start":start,"series_names":names,"source_units":units,
                          "structural_rows":[int(i)+1 for i in d.index[~date_rows]]}
        parts=[]
        for col in d.columns[1:]:
            if not names[col].strip():continue
            parts.append(pd.DataFrame({"period_raw":d.loc[date_rows,0],"raw_value":d.loc[date_rows,col],
                "commodity_raw":names[col],"unit_raw":units[col],"raw_row":d.index[date_rows]+1,
                "raw_column":int(col)+1,"statistic_kind":"price" if m["table"]=="Monthly Prices" else "index"}))
        VIEWS[ident]=pd.concat(parts,ignore_index=True)
    elif m["role"]=="observations":
        # Long tables preserve their original columns; aliases only in separate view.
        VIEWS[ident]=d
view_inventory=pd.DataFrame([{"id":ident,"observation_rows":len(view),
    "grain":"wide-cell observation projection" if "raw_value" in view.columns else "raw long row"} for ident,view in VIEWS.items()])
show(view_inventory,50)
schema_drift=inventory.groupby(["source","format"],dropna=False).agg(tables=("id","size"),fingerprints=("schema_fingerprint","nunique")).reset_index()
show(schema_drift)


,id,column_position,raw_column,normalized_candidate,storage_dtype,inferred_dtype,native_types,numeric_count,...,duplicate_header,blank_header,normalized_collision,leading_zero_count,encoding_suspect_count,header_encoding_suspect,field_role,examples
0,faostat/price.csv,1,Domain Code,domain_code,object,text/empty,{'lexical_str': 15334},0,...,False,False,False,0,0,False,provenance,['PP']
1,faostat/price.csv,2,Domain,domain,object,text/empty,{'lexical_str': 15334},0,...,False,False,False,0,0,False,provenance,['Producer Prices']
2,faostat/price.csv,3,Area Code (M49),area_code_m49,object,numeric,{'lexical_str': 15334},15334,...,False,False,False,1617,0,False,geography,"['004', '008', '012', '024']"
3,faostat/price.csv,4,Area,area,object,text/empty,{'lexical_str': 15334},0,...,False,False,False,0,0,False,geography,"['Afghanistan', 'Albania', 'Algeria', 'Angola']"
4,faostat/price.csv,5,Element Code,element_code,object,numeric,{'lexical_str': 15334},15334,...,False,False,False,0,0,False,measure/unit,"['5530', '5531', '5539', '5532']"
5,faostat/price.csv,6,Element,element,object,text/empty,{'lexical_str': 15334},0,...,False,False,False,0,0,False,measure/unit,"['Producer Price (LCU/tonne)', 'Producer Price (SLC/tonne)', 'Producer Price Index (2014-20..."
6,faostat/price.csv,7,Item Code (CPC),item_code_cpc,object,numeric,{'lexical_str': 15334},15334,...,False,False,False,15334,0,False,measure/unit,['0113']
7,faostat/price.csv,8,Item,item,object,text/empty,{'lexical_str': 15334},0,...,False,False,False,0,0,False,measure/unit,['Rice']
8,faostat/price.csv,9,Year Code,year_code,object,numeric,{'lexical_str': 15334},15334,...,False,False,False,0,0,False,time,"['1993', '1994', '1995', '1996']"
9,faostat/price.csv,10,Year,year,object,numeric,{'lexical_str': 15334},15334,...,False,False,False,0,0,False,time,"['1993', '1994', '1995', '1996']"


Displayed 18/1053 rows; full table remains in memory.


,id,column_position,raw_column,normalized_candidate,storage_dtype,inferred_dtype,native_types,numeric_count,...,duplicate_header,blank_header,normalized_collision,leading_zero_count,encoding_suspect_count,header_encoding_suspect,field_role,examples
2,faostat/price.csv,3,Area Code (M49),area_code_m49,object,numeric,{'lexical_str': 15334},15334,...,False,False,False,1617,0,False,geography,"['004', '008', '012', '024']"
6,faostat/price.csv,7,Item Code (CPC),item_code_cpc,object,numeric,{'lexical_str': 15334},15334,...,False,False,False,15334,0,False,measure/unit,['0113']
18,faostat/production_world.csv,3,Area Code (M49),area_code_m49,object,numeric,{'lexical_str': 22738},22738,...,False,False,False,2529,0,False,geography,"['004', '008', '012', '024']"
22,faostat/production_world.csv,7,Item Code (CPC),item_code_cpc,object,numeric,{'lexical_str': 22738},22738,...,False,False,False,22738,0,False,measure/unit,['0113']
33,faostat/supply_utilization.csv,3,Area Code (M49),area_code_m49,object,numeric,{'lexical_str': 36380},36380,...,False,False,False,4294,0,False,geography,"['004', '008', '012', '024']"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209,nso/V06.17.csv,1,"T?nh, thành ph?",t_nh_thanh_ph,object,text/empty,{'lexical_str': 72},0,...,False,False,False,0,41,True,geography,"['C? NU?C', 'Ð?ng b?ng sông H?ng', 'Hà N?i', 'Hà Tây']"
239,nso/V06.17.csv,31,So b? 2024,so_b_2024,object,numeric,{'lexical_str': 72},70,...,False,False,False,0,0,True,time,"['68.8', '66.9', '62.8', '..']"
240,nso/V06.18.csv,1,"T?nh, thành ph?",t_nh_thanh_ph,object,text/empty,{'lexical_str': 72},0,...,False,False,False,0,41,True,geography,"['C? NU?C', 'Ð?ng b?ng sông H?ng', 'Hà N?i', 'Hà Tây']"
270,nso/V06.18.csv,31,So b? 2024,so_b_2024,object,numeric,{'lexical_str': 72},70,...,False,False,False,0,0,True,time,"['20333.9', '3154.2', '508.7', '..']"


Displayed 35/137 rows; full table remains in memory.


,id,schema_fingerprint,organization,raw_rows,nonstructural_rows,year_fields,year_span,coverage_fingerprint,coverage_labels,group
0,nso/V06.12.csv,d075850e303d8e13,statistic_year_measure_table,71,70,0,Nam values,571d84f4533c6759,2,schema_group_01
1,nso/V06.13.csv,f845e3b32c566892,geography_year_matrix,72,71,30,1995 → So b? 2024,0c07a9a4e814b5e2,71,schema_group_02
2,nso/V06.14.csv,f845e3b32c566892,geography_year_matrix,72,71,30,1995 → So b? 2024,0c07a9a4e814b5e2,71,schema_group_02
3,nso/V06.15.csv,f845e3b32c566892,geography_year_matrix,72,71,30,1995 → So b? 2024,0c07a9a4e814b5e2,71,schema_group_02
4,nso/V06.16.csv,f845e3b32c566892,geography_year_matrix,72,71,30,1995 → So b? 2024,0c07a9a4e814b5e2,71,schema_group_02
5,nso/V06.17.csv,f845e3b32c566892,geography_year_matrix,72,71,30,1995 → So b? 2024,0c07a9a4e814b5e2,71,schema_group_02
6,nso/V06.18.csv,f845e3b32c566892,geography_year_matrix,72,71,30,1995 → So b? 2024,0c07a9a4e814b5e2,71,schema_group_02
7,nso/V06.19.csv,f845e3b32c566892,geography_year_matrix,38,37,30,1995 → So b? 2024,1619f3df9cc6ea44,37,schema_group_02
8,nso/V06.20.csv,f845e3b32c566892,geography_year_matrix,38,37,30,1995 → So b? 2024,1619f3df9cc6ea44,37,schema_group_02
9,nso/V06.21.csv,f845e3b32c566892,geography_year_matrix,38,37,30,1995 → So b? 2024,1619f3df9cc6ea44,37,schema_group_02


,id,observation_rows,grain
0,faostat/price.csv,15334,raw long row
1,faostat/production_world.csv,22738,raw long row
2,faostat/supply_utilization.csv,36380,raw long row
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,1226470,raw long row
4,nso/V06.12.csv,560,wide-cell observation projection
...,...,...,...
17,thitruongnongsan/price_luagao.xlsx::price_lua_gao,20394,raw long row
18,usda/Export-prices-Thailand-Vietnam-India-and-Pakistan.csv,13131,raw long row
19,usda/usda.xls::html_table_1,990,wide-cell observation projection
20,world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Prices,56232,wide-cell observation projection


,source,format,tables,fingerprints
0,faostat,csv,9,8
1,nso,csv,13,2
2,thitruongnongsan,xlsx,1,1
3,usda,csv,1,1
4,usda,html (.xls extension),1,1
5,world_bank,xlsx,5,5


**Diễn giải schema:** nhiều numeric-looking code là identifier, cần giữ string (M49/CPC có leading zero, CPC có dấu chấm và prefix chữ). Column unit trống của price index không chứng minh giá trị thiếu đơn vị vật lý. Trong workbook World Bank, tên series ở header nhiều dòng và sheet Description/Index Weights là metadata, không phải observation. NSO cùng fingerprint có thể khác phạm vi địa phương/mùa vụ.


## STEP 3 — Missing Value Analysis

python_null, empty_string, whitespace_only và từng token được đếm tách biệt. token_candidates là dấu hiệu cần diễn giải, không tự chuyển thành NULL. zero_numeric luôn là số 0 quan sát được. Row toàn blank thuộc structural evidence; STEP 7 mới đề xuất xử lý.

missing_profile chứa từng cột Raw; observation_missing_profile dùng các projection đã loại header/footer cấu trúc khỏi mẫu số. grouped_missing phân tích theo measure/địa phương/statistic_kind/period/flag. Các nhóm có mức thiếu khác nhau sẽ hiển thị trước, không gộp các loại missing thành “missing đã xác nhận”.


In [4]:
def missing_stats(s):
    if isinstance(s.dtype,pd.CategoricalDtype):
        counts=s.value_counts(dropna=True,sort=False)
        values=pd.Series(counts.index,dtype="string")
        weights=counts.to_numpy()
        stripped=values.str.strip()
        numeric,_=numeric_probe(values)
        weighted=lambda mask:int(np.sum(weights[np.asarray(mask.fillna(False),dtype=bool)]))
        stats={"python_null":int(s.isna().sum()),"empty_string":weighted(values.eq("")),
            "whitespace_only":weighted(values.ne("")&stripped.eq("")),
            "token_candidates":weighted(stripped.isin(MISSING_TOKEN_CANDIDATES)),
            "zero_numeric":weighted(numeric.eq(0))}
        stats.update({"token_"+repr(token):weighted(stripped.eq(token)) for token in MISSING_TOKEN_CANDIDATES})
        stats["blank_or_null"]=stats["python_null"]+stats["empty_string"]+stats["whitespace_only"]
        stats["unresolved_missing_candidates"]=stats["blank_or_null"]+stats["token_candidates"]
        stats["denominator_cells"]=len(s)
        stats["candidate_rate_pct"]=100*stats["unresolved_missing_candidates"]/len(s) if len(s) else 0.
        return stats
    t=text_series(s)
    stripped=t.str.strip()
    numeric,_=numeric_probe(s)
    stats={"python_null":int(s.isna().sum()),"empty_string":int(t.eq("").fillna(False).sum()),
        "whitespace_only":int((t.ne("").fillna(False)&stripped.eq("").fillna(False)).sum()),
        "token_candidates":int(stripped.isin(MISSING_TOKEN_CANDIDATES).sum()),
        "zero_numeric":int(numeric.eq(0).sum())}
    stats.update({"token_"+repr(token):int(stripped.eq(token).fillna(False).sum()) for token in MISSING_TOKEN_CANDIDATES})
    stats["blank_or_null"]=stats["python_null"]+stats["empty_string"]+stats["whitespace_only"]
    stats["unresolved_missing_candidates"]=stats["blank_or_null"]+stats["token_candidates"]
    stats["denominator_cells"]=len(s)
    stats["candidate_rate_pct"]=100*stats["unresolved_missing_candidates"]/len(s) if len(s) else 0.
    return stats

missing_rows=[]
structural_rows=[]
for ident,entry in DATA.items():
    d=entry["raw"]; blank=structural_mask(d)
    structural_rows.append({"id":ident,"all_blank_records":int(blank.sum()),
        "raw_record_positions":str((d.index[blank]+(2 if entry["meta"]["header_rows"] else 1)).tolist()[:8]),
        "role":entry["meta"]["role"]})
    for pos,c in enumerate(d.columns):
        missing_rows.append({"id":ident,"column":str(c),"column_position":pos+1,**missing_stats(d.iloc[:,pos])})
missing_profile=pd.DataFrame(missing_rows)
structural_profile=pd.DataFrame(structural_rows)
observation_missing_rows=[];grouped_rows=[]
for ident,v in VIEWS.items():
    value_field=next((c for c in ("raw_value","Value","VALUE","Giá") if c in v.columns),None)
    if value_field is None:continue
    observation_missing_rows.append({"id":ident,"field":value_field,**missing_stats(v[value_field])})
    group_fields=[c for c in ("Element","Attribute","Months","REFERENCE_PERIOD_DESCRIPTION","Flag",
        "geography_raw","statistic_kind","commodity_raw","Tên_mặt_hàng","Thị_trường") if c in v.columns]
    for field in group_fields:
        # Categorical grouping remains full-data; observed=True excludes synthetic empty groups.
        for value,g in v.groupby(field,dropna=False,observed=True,sort=True):
            stats=missing_stats(g[value_field])
            grouped_rows.append({"id":ident,"group_field":field,"group_value":str(value),**stats})
observation_missing_profile=pd.DataFrame(observation_missing_rows,columns=["id","field"]+list(missing_stats(pd.Series([],dtype="string"))))
grouped_missing=pd.DataFrame(grouped_rows)
show(observation_missing_profile,50)
show(structural_profile.loc[structural_profile.all_blank_records.gt(0)],30)
if len(grouped_missing):
    show(grouped_missing.sort_values("candidate_rate_pct",ascending=False),25)
file_missing_summary=missing_profile.groupby("id").agg(
    raw_cells=("denominator_cells","sum"),null_cells=("python_null","sum"),empty_cells=("empty_string","sum"),
    whitespace_cells=("whitespace_only","sum"),token_cells=("token_candidates","sum"),zero_cells=("zero_numeric","sum")).reset_index()
show(file_missing_summary,50)


,id,field,python_null,empty_string,whitespace_only,token_candidates,zero_numeric,token_'..',...,token_'N/A',token_'-',token_'...',token_'…',blank_or_null,unresolved_missing_candidates,denominator_cells,candidate_rate_pct
0,faostat/price.csv,Value,0,0,0,0,2,0,...,0,0,0,0,0,0,15334,0.000000
1,faostat/production_world.csv,Value,0,229,0,0,572,0,...,0,0,0,0,229,229,22738,1.007125
2,faostat/supply_utilization.csv,Value,0,257,0,0,8660,0,...,0,0,0,0,257,257,36380,0.706432
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Value,0,0,0,0,133948,0,...,0,0,0,0,0,0,1226470,0.000000
4,nso/V06.12.csv,raw_value,0,0,0,0,0,0,...,0,0,0,0,0,0,560,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,thitruongnongsan/price_luagao.xlsx::price_lua_gao,Giá,0,0,0,0,29,0,...,0,0,0,0,0,0,20394,0.000000
18,usda/Export-prices-Thailand-Vietnam-India-and-Pakistan.csv,VALUE,0,0,0,4594,0,0,...,0,0,0,0,0,4594,13131,34.985911
19,usda/usda.xls::html_table_1,raw_value,0,0,0,0,140,0,...,0,0,0,0,0,0,990,0.000000
20,world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Prices,raw_value,0,0,0,6405,0,0,...,0,0,219,6186,0,6405,56232,11.390312


,id,all_blank_records,raw_record_positions,role
9,nso/V06.12.csv,1,[72],observations
10,nso/V06.13.csv,1,[73],observations
11,nso/V06.14.csv,1,[73],observations
12,nso/V06.15.csv,1,[73],observations
13,nso/V06.16.csv,1,[73],observations
14,nso/V06.17.csv,1,[73],observations
15,nso/V06.18.csv,1,[73],observations
16,nso/V06.19.csv,1,[39],observations
17,nso/V06.20.csv,1,[39],observations
18,nso/V06.21.csv,1,[39],observations


,id,group_field,group_value,python_null,empty_string,whitespace_only,token_candidates,zero_numeric,...,token_'N/A',token_'-',token_'...',token_'…',blank_or_null,unresolved_missing_candidates,denominator_cells,candidate_rate_pct
43,faostat/supply_utilization.csv,Flag,M,0,257,0,0,0,...,0,0,0,0,257,257,257,100.000000
26,faostat/production_world.csv,Flag,M,0,229,0,0,0,...,0,0,0,0,229,229,229,100.000000
765,nso/V06.24.csv,geography_raw,H?u Giang,0,0,0,30,0,...,0,0,0,0,0,30,30,100.000000
621,nso/V06.22.csv,geography_raw,H?u Giang,0,0,0,30,0,...,0,0,0,0,0,30,30,100.000000
693,nso/V06.23.csv,geography_raw,H?u Giang,0,0,0,30,0,...,0,0,0,0,0,30,30,100.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,nso/V06.20.csv,geography_raw,Qu?ng Nam,0,0,0,20,0,...,0,0,0,0,0,20,30,66.666667
585,nso/V06.21.csv,geography_raw,Qu?ng Nam,0,0,0,20,0,...,0,0,0,0,0,20,30,66.666667
805,nso/V06.24.csv,geography_raw,Vinh Long,0,0,0,20,0,...,0,0,0,0,0,20,30,66.666667
661,nso/V06.22.csv,geography_raw,Vinh Long,0,0,0,20,0,...,0,0,0,0,0,20,30,66.666667


Displayed 25/1147 rows; full table remains in memory.


,id,raw_cells,null_cells,empty_cells,whitespace_cells,token_cells,zero_cells
0,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Eleme...,20,0,0,0,0,0
1,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Flags...,8,0,0,0,0,0
2,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_ItemC...,1716,0,0,0,0,0
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Partn...,765,0,0,0,0,0
4,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Repor...,696,0,0,0,0,0
...,...,...,...,...,...,...,...
24,usda/usda.xls::html_table_1,1050,0,14,0,0,140
25,world_bank/CMO-Historical-Data-Monthly.xlsx::Description,35061,34819,0,0,0,0
26,world_bank/CMO-Historical-Data-Monthly.xlsx::Index Weights,23092,22887,0,0,0,0
27,world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Indices,110322,96832,0,6,0,0


## STEP 4 — Duplicate & Business Key Analysis

Exact duplicate excess là số bản ghi dư (duplicated keep=first), duplicate participants gồm mọi bản ghi trong nhóm (keep=False). Key conflict là cùng key nhưng khác ít nhất một trường Raw; value-conflict riêng chỉ xét measure. Không chọn winner, không xóa.

Keys thử hai mức khi có ích (có/không nguồn hoặc unit). Row blank cấu trúc được tách trước khi kiểm tra NSO observation keys; số dòng Raw vẫn được báo. Release overlap chỉ được kết luận khi có version/period evidence; cùng schema chưa đủ để deduplicate hai dataset.


In [5]:
KEYS={}
for ident,v in VIEWS.items():
    source=DATA[ident]["meta"]["source"]
    keys=[]
    if source=="faostat":
        if "Reporter Country Code" in v:
            base=["Reporter Country Code","Partner Country Code","Item Code","Element Code","Year"]
        else:
            base=[c for c in ["Domain Code","Area Code (M49)","Item Code (CPC)","Element Code","Year","Months Code"] if c in v]
        if base:
            keys.append(base)
            if "Unit" in v: keys.append(base+["Unit"])
    elif source=="nso":
        keys=[["geography_raw","period_raw","raw_column","statistic_kind"]]
    elif source=="usda" and "Attribute" in v:
        keys=[["Commodity","Attribute","Country","period_raw"]]
    elif source=="usda":
        keys=[[c for c in ["TABLE_NUMBER","COMMODITY_DESCRIPTION","CLASS_DESCRIPTION","YEAR","REFERENCE_PERIOD_DESCRIPTION",
                          "LOCATION_DESCRIPTION","STATISTIC_DESCRIPTION","UNIT_DESCRIPTION"] if c in v]]
    elif source=="world_bank":
        keys=[["commodity_raw","period_raw","statistic_kind"]]
    elif source=="thitruongnongsan":
        base=[c for c in ["Tên_mặt_hàng","Thị_trường","Loại_giá","Đơn_vị_tính","Loại_tiền","Ngày"] if c in v]
        keys=[base,base+["Nguồn"]]
    KEYS[ident]=keys

KEY_DATA=dict(VIEWS)
for ident,entry in DATA.items():
    if entry["meta"]["role"]=="reference_data" and len(entry["raw"].columns):
        KEYS[ident]=[[entry["raw"].columns[0]]]
        KEY_DATA[ident]=entry["raw"]
duplicate_rows=[];key_rows=[];key_examples={}
for ident,entry in DATA.items():
    d=entry["raw"]
    duplicate_rows.append({"id":ident,"scope":"raw table","rows":len(d),
        "exact_duplicate_excess":int(d.duplicated().sum()),
        "exact_duplicate_participants":int(d.duplicated(keep=False).sum())})
for ident,v in KEY_DATA.items():
    comparison=[c for c in v.columns if c!="raw_row"]
    distinct=v.drop_duplicates(subset=comparison)
    value_field=next((c for c in ("raw_value","Value","VALUE","Giá") if c in v),None)
    for number,key in enumerate(KEYS.get(ident,[]),1):
        mask=v.duplicated(key,keep=False)
        variants=distinct.loc[distinct.duplicated(key,keep=False)]
        conflict_groups=len(variants[key].drop_duplicates())
        value_conflicts=0
        if value_field and mask.any():
            nvalues=v.loc[mask,key+[value_field]].drop_duplicates()
            value_conflicts=len(nvalues.loc[nvalues.duplicated(key,keep=False),key].drop_duplicates())
        missing_key=pd.Series(False,index=v.index)
        for c in key:
            missing_key |= v[c].isna() | text_series(v[c]).fillna("").str.strip().eq("") | text_series(v[c]).str.strip().isin(MISSING_TOKEN_CANDIDATES)
        key_rows.append({"id":ident,"candidate":number,"key":" + ".join(key),"rows":len(v),
            "key_incomplete_rows":int(missing_key.sum()),"duplicate_participants":int(mask.sum()),
            "duplicate_excess":int(v.duplicated(key).sum()),"conflict_groups":conflict_groups,
            "value_conflict_groups":value_conflicts})
        if mask.any():
            key_examples[(ident,number)]=v.loc[mask].head(6).copy()
duplicates=pd.DataFrame(duplicate_rows)
key_analysis=pd.DataFrame(key_rows)
show(duplicates,50)
show(key_analysis,50)
for (ident,number),example in key_examples.items():
    display(Markdown(f"**Conflicting/duplicate example: {ident}, candidate {number}**"))
    display(example)
release_analysis=pd.DataFrame([{"id":ident,
    "release_fields":", ".join(c for c in entry["raw"].columns if re.search(r"release|version|ingest|updated|revision",str(c),re.I)),
    "snapshot":entry["meta"]["content_sha256"][:16],
    "limitation":"No winner policy inferred from file order; missing release identity needs source review"}
    for ident,entry in DATA.items() if entry["meta"]["role"]=="observations"])
same_content_files=inventory[["file","content_sha256"]].drop_duplicates()
same_content_files=same_content_files.loc[same_content_files.duplicated("content_sha256",keep=False)]
show(release_analysis)
print("Byte-identical files across inventory:",len(same_content_files))


,id,scope,rows,exact_duplicate_excess,exact_duplicate_participants
0,faostat/price.csv,raw table,15334,0,0
1,faostat/production_world.csv,raw table,22738,0,0
2,faostat/supply_utilization.csv,raw table,36380,0,0
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,raw table,1226470,0,0
4,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Eleme...,raw table,10,0,0
...,...,...,...,...,...
25,world_bank/CMO-Historical-Data-Monthly.xlsx::AFOSHEET,raw table,0,0,0
26,world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Prices,raw table,798,0,0
27,world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Indices,raw table,1362,561,562
28,world_bank/CMO-Historical-Data-Monthly.xlsx::Description,raw table,403,291,292


,id,candidate,key,rows,key_incomplete_rows,duplicate_participants,duplicate_excess,conflict_groups,value_conflict_groups
0,faostat/price.csv,1,Domain Code + Area Code (M49) + Item Code (CPC) + Element Code + Year + Months Code,15334,0,0,0,0,0
1,faostat/price.csv,2,Domain Code + Area Code (M49) + Item Code (CPC) + Element Code + Year + Months Code + Unit,15334,3136,0,0,0,0
2,faostat/production_world.csv,1,Domain Code + Area Code (M49) + Item Code (CPC) + Element Code + Year,22738,0,0,0,0,0
3,faostat/production_world.csv,2,Domain Code + Area Code (M49) + Item Code (CPC) + Element Code + Year + Unit,22738,0,0,0,0,0
4,faostat/supply_utilization.csv,1,Domain Code + Area Code (M49) + Item Code (CPC) + Element Code + Year,36380,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
27,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Eleme...,1,Element Code,10,0,0,0,0,0
28,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Flags...,1,Flag,4,0,0,0,0,0
29,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_ItemC...,1,Item Code,572,0,0,0,0,0
30,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Partn...,1,Partner Country Code,255,0,0,0,0,0


**Conflicting/duplicate example: thitruongnongsan/price_luagao.xlsx::price_lua_gao, candidate 1**

,Tên_mặt_hàng,Thị_trường,Loại_giá,Đơn_vị_tính,Loại_tiền,Nguồn,Ngày,Giá
3891,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/12/2025 12:00:00 AM,5600
3892,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/12/2025 12:00:00 AM,5600
3906,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/5/2025 12:00:00 AM,5600
3907,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/5/2025 12:00:00 AM,5600
3950,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,2/10/2025 12:00:00 AM,5500
3951,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,2/10/2025 12:00:00 AM,5600


**Conflicting/duplicate example: thitruongnongsan/price_luagao.xlsx::price_lua_gao, candidate 2**

,Tên_mặt_hàng,Thị_trường,Loại_giá,Đơn_vị_tính,Loại_tiền,Nguồn,Ngày,Giá
3891,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/12/2025 12:00:00 AM,5600
3892,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/12/2025 12:00:00 AM,5600
3906,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/5/2025 12:00:00 AM,5600
3907,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,3/5/2025 12:00:00 AM,5600
3950,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,2/10/2025 12:00:00 AM,5500
3951,Lúa IR 50404 (lúa tươi),An Giang,Thương lái thu mua,VNĐ/Kg,VNĐ,CTV địa phương,2/10/2025 12:00:00 AM,5600


**Conflicting/duplicate example: usda/Export-prices-Thailand-Vietnam-India-and-Pakistan.csv, candidate 1**

,TABLE_NAME,TABLE_NUMBER,COMMODITY_DESCRIPTION,CLASS_DESCRIPTION,YEAR,REFERENCE_PERIOD_DESCRIPTION,LOCATION_DESCRIPTION,STATISTIC_DESCRIPTION,VALUE,UNIT_DESCRIPTION
7629,"Milled rice export price quotes, India, 1997/98 to present",27,RICE,5-PERCENT BROKEN,2000,August,INDIA,FREIGHT ON BOARD PRICE,264,DOLLARS PER METRIC TON
7630,"Milled rice export price quotes, India, 1997/98 to present",27,RICE,10-PERCENT BROKEN,2000,August,INDIA,FREIGHT ON BOARD PRICE,257,DOLLARS PER METRIC TON
7631,"Milled rice export price quotes, India, 1997/98 to present",27,RICE,15-PERCENT BROKEN,2000,August,INDIA,FREIGHT ON BOARD PRICE,249,DOLLARS PER METRIC TON
7632,"Milled rice export price quotes, India, 1997/98 to present",27,RICE,25-PERCENT BROKEN,2000,August,INDIA,FREIGHT ON BOARD PRICE,237,DOLLARS PER METRIC TON
7633,"Milled rice export price quotes, India, 1997/98 to present",27,RICE,100-PERCENT BROKEN,2000,August,INDIA,FREIGHT ON BOARD PRICE,NA,DOLLARS PER METRIC TON
7634,"Milled rice export price quotes, India, 1997/98 to present",27,RICE,5-PERCENT PARBOILED,2000,August,INDIA,FREIGHT ON BOARD PRICE,246,DOLLARS PER METRIC TON


,id,release_fields,snapshot,limitation
0,faostat/price.csv,,c6d49e6a1430866c,No winner policy inferred from file order; missing release identity needs source review
1,faostat/production_world.csv,,6a23d820d61e990d,No winner policy inferred from file order; missing release identity needs source review
2,faostat/supply_utilization.csv,,3a0fa590b9fd8c3a,No winner policy inferred from file order; missing release identity needs source review
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,,697bdc393bc0283b,No winner policy inferred from file order; missing release identity needs source review
4,nso/V06.12.csv,,22f59d3870d02142,No winner policy inferred from file order; missing release identity needs source review
5,nso/V06.13.csv,,4aeba037c50f5ff2,No winner policy inferred from file order; missing release identity needs source review
6,nso/V06.14.csv,,8009fef207bec209,No winner policy inferred from file order; missing release identity needs source review
7,nso/V06.15.csv,,a3c8dc9ab522550f,No winner policy inferred from file order; missing release identity needs source review
8,nso/V06.16.csv,,57aeb150faa8b93b,No winner policy inferred from file order; missing release identity needs source review
9,nso/V06.17.csv,,fbbb470e19ba5295,No winner policy inferred from file order; missing release identity needs source review


Displayed 18/22 rows; full table remains in memory.
Byte-identical files across inventory: 0


## STEP 5 — Data Type & Range Validation

Numeric parsing cho phép dấu phẩy chỉ khi đúng dạng nhóm nghìn; không đoán dấu phẩy thập phân. Đếm tokens riêng khỏi genuine parsing failures. Negative không mặc định sai. Outliers là tín hiệu IQR theo measure/unit/commodity; chỉ FLAG, không xóa, không dùng làm giới hạn domain. Nhiều measure có trend và geography scale nên IQR cần review.

Year range dùng AS_OF_DATE và cận dưới cấu hình 1900; marketing-year label được kiểm tra end=start+1. Date format cho thị trường là US month/day theo các giá trị có day>12; không suy từ locale kernel. Identifier morphology chỉ kiểm tra format và dictionary khi có source lookup; không thay thế country/code registry.


In [6]:
numeric_rows=[];range_examples={};period_rows=[];category_rows=[];identifier_rows=[]
for ident,v in VIEWS.items():
    value_field=next((c for c in ("raw_value","Value","VALUE","Giá") if c in v),None)
    if value_field:
        t=text_series(v[value_field]);n,commas=numeric_probe(v[value_field])
        recognized=t.str.strip().isin(MISSING_TOKEN_CANDIDATES)|t.fillna("").str.strip().eq("")
        failures=n.isna()&~recognized&v[value_field].notna()
        nonfinite=n.notna()&~np.isfinite(n.astype(float))
        group_fields=[c for c in ("Element","Unit","Item","Attribute","Unit Description","raw_column","statistic_kind",
                                  "commodity_raw","CLASS_DESCRIPTION","LOCATION_DESCRIPTION","UNIT_DESCRIPTION",
                                  "Tên_mặt_hàng","Đơn_vị_tính","Loại_tiền") if c in v]
        # raw_column for WB is lineage, series identity already provides grouping.
        if DATA[ident]["meta"]["source"]=="world_bank":group_fields=["commodity_raw","unit_raw","statistic_kind"]
        tmp=v[group_fields].copy();tmp["_numeric"]=n
        if group_fields:
            grp=tmp.groupby(group_fields,observed=True,dropna=False,sort=False)["_numeric"]
            q1=grp.transform("quantile",q=.25);q3=grp.transform("quantile",q=.75)
            group_count=grp.transform("count")
        else:
            q1=pd.Series(n.quantile(.25),index=n.index);q3=pd.Series(n.quantile(.75),index=n.index)
            group_count=pd.Series(n.notna().sum(),index=n.index)
        spread=q3-q1
        outlier=n.notna()&group_count.ge(8)&spread.gt(0)&((n<q1-1.5*spread)|(n>q3+1.5*spread))
        precision=t.str.extract(r"\.(\d+)(?:[eE][+-]?\d+)?$",expand=False).str.len().value_counts().sort_index().to_dict()
        numeric_rows.append({"id":ident,"field":value_field,"observations":len(v),"numeric_valid":int(n.notna().sum()),
            "missing_candidates":int(recognized.sum()+v[value_field].isna().sum()-int((recognized&v[value_field].isna()).sum())),
            "parse_failures":int(failures.sum()),"comma_grouped":int(commas.sum()),"nonfinite":int(nonfinite.sum()),
            "negative":int(n.lt(0).sum()),"zero":int(n.eq(0).sum()),"min":n.min(),"max":n.max(),
            "iqr_flagged":int(outlier.sum()),"iqr_grouping":", ".join(group_fields),"decimal_precision_counts":str(precision)})
        for issue,mask in [("parse_failure",failures),("negative",n.lt(0)),("zero",n.eq(0)),("iqr",outlier),("comma_grouped",commas)]:
            if mask.any():range_examples[(ident,issue)]=v.loc[mask].head(5).copy()
        if group_fields:
            for vals,g in tmp.groupby(group_fields,observed=True,dropna=False,sort=True):
                vals=vals if isinstance(vals,tuple) else (vals,)
                category_rows.append({"id":ident,"dimension":"numeric group",
                    "category":str(dict(zip(group_fields,vals))),"rows":len(g),"numeric_rows":int(g._numeric.notna().sum()),
                    "negative":int(g._numeric.lt(0).sum()),"min":g._numeric.min(),"max":g._numeric.max()})
    for c in v.columns:
        if str(c) in {"Year","YEAR","year_candidate"}:
            n,_=numeric_probe(v[c]);bad=n.isna()|n.lt(1900)|n.gt(AS_OF_DATE.year+1)|n.mod(1).ne(0)
            period_rows.append({"id":ident,"field":str(c),"type":"year candidate","min":n.min(),"max":n.max(),
                               "invalid":int(bad.sum()),"missing_periods":"not inferred without expected coverage"})
        elif str(c)=="period_raw":
            ts=text_series(v[c])
            if ts.str.fullmatch(r"\d{4}M\d{2}",na=False).all():
                dates=pd.to_datetime(ts,format="%YM%m",errors="coerce")
                unique=dates.dropna().drop_duplicates().sort_values()
                expected=pd.date_range(unique.min(),unique.max(),freq="MS") if len(unique) else pd.DatetimeIndex([])
                missing=expected.difference(pd.DatetimeIndex(unique))
                period_rows.append({"id":ident,"field":c,"type":"calendar month","min":str(dates.min()),
                    "max":str(dates.max()),"invalid":int(dates.isna().sum()),"missing_periods":str([str(x.date()) for x in missing])})
            elif ts.str.fullmatch(r"\d{4}/\d{4}",na=False).all():
                starts=pd.to_numeric(ts.str[:4]);ends=pd.to_numeric(ts.str[-4:])
                bad=ends.ne(starts+1)|starts.lt(1900)|ends.gt(AS_OF_DATE.year+1)
                period_rows.append({"id":ident,"field":c,"type":"marketing/crop year; exact start month unknown",
                    "min":ts.min(),"max":ts.max(),"invalid":int(bad.sum()),"missing_periods":"period labels only"})
        elif candidate_name(c) in {"ngay","date"}:
            dates=pd.to_datetime(v[c],format="%m/%d/%Y %I:%M:%S %p",errors="coerce")
            bad=dates.isna()|dates.gt(AS_OF_DATE+pd.Timedelta(days=1))
            period_rows.append({"id":ident,"field":str(c),"type":"month/day/year timestamp; timezone absent",
                "min":str(dates.min()),"max":str(dates.max()),"invalid":int(bad.sum()),
                "missing_periods":"daily market coverage not assumed; no expected trading calendar"})
    for c in v.columns:
        if re.search(r"code|flag|unit|element|attribute|months|period_description|loai_gia|don_vi",candidate_name(c)):
            vc=v[c].value_counts(dropna=False)
            category_rows.extend({"id":ident,"dimension":str(c),"category":str(val),"rows":int(count)}
                                 for val,count in vc.items())
        if re.search(r"code|m49|cpc",str(c),re.I):
            t=text_series(v[c]).str.strip()
            present=t.notna()&t.ne("")
            pattern=r"'?\d{3}" if "m49" in str(c).lower() else (r"'?[A-Za-z0-9]+(?:\.\d+)?" if "cpc" in str(c).lower() else r"[A-Za-z0-9]+" if "domain" in str(c).lower() else r"\d+")
            bad=present&~t.str.fullmatch(pattern,na=False)
            identifier_rows.append({"id":ident,"field":str(c),"pattern_candidate":pattern,"format_suspects":int(bad.sum()),
                "empty":int((~present).sum()),"examples":str(t.loc[bad].unique().tolist()[:5]),
                "validation_scope":"morphology only; registry validation pending unless local dictionary"})
numeric_profile=pd.DataFrame(numeric_rows,columns=["id","field","observations","numeric_valid","missing_candidates","parse_failures","comma_grouped","nonfinite","negative","zero","min","max","iqr_flagged","iqr_grouping","decimal_precision_counts"])
period_profile=pd.DataFrame(period_rows,columns=["id","field","type","min","max","invalid","missing_periods"])
category_profile=pd.DataFrame(category_rows)
identifier_profile=pd.DataFrame(identifier_rows)
show(numeric_profile,50)
show(period_profile,50)
show(identifier_profile.loc[identifier_profile.format_suspects.gt(0)],25) if len(identifier_profile) else None
for issue in ("negative","zero","parse_failure"):
    for (ident,kind),examples in range_examples.items():
        if kind==issue:
            print(ident,kind);display(examples)


,id,field,observations,numeric_valid,missing_candidates,parse_failures,comma_grouped,nonfinite,negative,zero,min,max,iqr_flagged,iqr_grouping,decimal_precision_counts
0,faostat/price.csv,Value,15334,15334,0,0,0,0,0,2,0.000000e+00,7.311370e+08,1911,"Element, Unit, Item","{np.int64(1): 4369, np.int64(2): 2818}"
1,faostat/production_world.csv,Value,22738,22509,229,0,0,0,0,572,0.000000e+00,2.178679e+08,2524,"Element, Unit, Item","{np.int64(1): 6278, np.int64(2): 467}"
2,faostat/supply_utilization.csv,Value,36380,36123,257,0,0,0,1516,8660,-6.425000e+06,2.144299e+08,6364,"Element, Unit, Item","{np.int64(1): 2569, np.int64(2): 16691}"
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Value,1226470,1226470,0,0,0,0,0,133948,0.000000e+00,5.004090e+06,214684,"Element, Unit, Item","{np.int64(1): 1005067, np.int64(2): 221403}"
4,nso/V06.12.csv,raw_value,560,560,0,0,0,0,0,0,8.630000e+01,4.509100e+04,16,"raw_column, statistic_kind",{np.int64(1): 560}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,thitruongnongsan/price_luagao.xlsx::price_lua_gao,Giá,20394,20394,0,0,0,0,0,29,0.000000e+00,8.000000e+04,920,"Tên_mặt_hàng, Đơn_vị_tính, Loại_tiền","{np.int64(1): 206, np.int64(2): 418}"
18,usda/Export-prices-Thailand-Vietnam-India-and-Pakistan.csv,VALUE,13131,8537,4594,0,0,0,0,0,9.675000e+01,2.000000e+03,141,"CLASS_DESCRIPTION, LOCATION_DESCRIPTION, UNIT_DESCRIPTION","{np.int64(1): 558, np.int64(2): 1026}"
19,usda/usda.xls::html_table_1,raw_value,990,990,0,0,597,0,0,140,0.000000e+00,4.506600e+04,0,"Attribute, Unit Description, raw_column",{np.int64(2): 66}
20,world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Prices,raw_value,56232,49827,6405,0,0,0,0,0,2.870000e-02,5.217905e+04,1138,"commodity_raw, unit_raw, statistic_kind","{np.int64(1): 6047, np.int64(2): 14086, np.int64(3): 2427, np.int64(4): 6220, np.int64(5): ..."


,id,field,type,min,max,invalid,missing_periods
0,faostat/price.csv,Year,year candidate,1991,2025,0,not inferred without expected coverage
1,faostat/production_world.csv,Year,year candidate,1961,2024,0,not inferred without expected coverage
2,faostat/supply_utilization.csv,Year,year candidate,2010,2023,0,not inferred without expected coverage
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Year,year candidate,1986,2024,0,not inferred without expected coverage
4,nso/V06.12.csv,year_candidate,year candidate,1990,2024,0,not inferred without expected coverage
...,...,...,...,...,...,...,...
17,thitruongnongsan/price_luagao.xlsx::price_lua_gao,Ngày,month/day/year timestamp; timezone absent,2021-01-04 00:00:00,2026-08-03 00:00:00,0,daily market coverage not assumed; no expected trading calendar
18,usda/Export-prices-Thailand-Vietnam-India-and-Pakistan.csv,YEAR,year candidate,1985,2025,0,not inferred without expected coverage
19,usda/usda.xls::html_table_1,period_raw,marketing/crop year; exact start month unknown,1960/1961,2025/2026,0,period labels only
20,world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Prices,period_raw,calendar month,1960-01-01 00:00:00,2025-12-01 00:00:00,0,[]


,id,field,pattern_candidate,format_suspects,empty,examples,validation_scope


faostat/supply_utilization.csv negative


,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
18,SCL,Supply Utilization Accounts (2010-),004,Afghanistan,5071,Stock Variation,0113,Rice,2012,2012,t,-10521.01,E,Estimated value,
26,SCL,Supply Utilization Accounts (2010-),004,Afghanistan,5071,Stock Variation,0113,Rice,2013,2013,t,-11214.4,E,Estimated value,
35,SCL,Supply Utilization Accounts (2010-),004,Afghanistan,5071,Stock Variation,0113,Rice,2014,2014,t,-15000,E,Estimated value,
43,SCL,Supply Utilization Accounts (2010-),004,Afghanistan,5071,Stock Variation,0113,Rice,2015,2015,t,-15000,E,Estimated value,
52,SCL,Supply Utilization Accounts (2010-),004,Afghanistan,5071,Stock Variation,0113,Rice,2016,2016,t,-15000,E,Estimated value,


...

,geography_raw,raw_row,raw_column,raw_value,period_raw,statistic_kind,year_candidate,provisional
15,Tây Nguyên,17,1995,0.0,1995,physical_quantity,1995,False
52,Tây Nguyên,17,1996,0.0,1996,physical_quantity,1996,False


thitruongnongsan/price_luagao.xlsx::price_lua_gao zero


,Tên_mặt_hàng,Thị_trường,Loại_giá,Đơn_vị_tính,Loại_tiền,Nguồn,Ngày,Giá
12244,Gạo NL 5% tấm,Kiên Giang,Thương lái thu mua,Vnđ/Kg,VNĐ,CTV địa phương,6/9/2023 12:00:00 AM,0
12276,Gạo NL 5% tấm,Kiên Giang,Thương lái thu mua,Vnđ/Kg,VNĐ,CTV địa phương,6/7/2023 12:00:00 AM,0
12386,Gạo NL 5% tấm,Kiên Giang,Thương lái thu mua,Vnđ/Kg,VNĐ,CTV địa phương,5/26/2023 12:00:00 AM,0
12418,Gạo NL 5% tấm,Kiên Giang,Thương lái thu mua,Vnđ/Kg,VNĐ,CTV địa phương,5/24/2023 12:00:00 AM,0
12493,Gạo NL 5% tấm,Kiên Giang,Thương lái thu mua,Vnđ/Kg,VNĐ,CTV địa phương,5/19/2023 12:00:00 AM,0


usda/usda.xls::html_table_1 zero


,Commodity,Attribute,Country,Unit Description,raw_row,period_raw,raw_value,raw_column
1,,Beginning Stocks,Vietnam,(1000 MT),3,1960/1961,0,1960/1961
12,,Ending Stocks,Vietnam,(1000 MT),14,1960/1961,0,1960/1961
16,,Beginning Stocks,Vietnam,(1000 MT),3,1961/1962,0,1961/1962
27,,Ending Stocks,Vietnam,(1000 MT),14,1961/1962,0,1961/1962
31,,Beginning Stocks,Vietnam,(1000 MT),3,1962/1963,0,1962/1963


**Diễn giải:** blank giá trị FAOSTAT đi kèm Flag M có thể xác nhận từ mô tả flag trong cùng Raw. NA của USDA và … của World Bank vẫn là source-specific missing proposals nếu chưa có chú giải chính thức. Số 0 giá thị trường cần xác minh nghiệp vụ; số 0 PSD hoặc trade được giữ. Header NSO bị lỗi là mất thông tin source, không dùng fuzzy mapping để “sửa” tên tỉnh.


## STEP 6 — Business Semantics

Sau profiling độc lập, chỉ đọc source metadata nằm trong Raw (NSO metadata.txt; lookup FAOSTAT; workbook Description/Index Weights). Không kế thừa quy tắc hoặc kết luận từ EDA/Silver cũ.

Milling Rate: tài liệu FAS chính thức nêu mức 62.5–62.7% cho Việt Nam; công thức bảng PSD lịch sử có factor 10000. STEP 8 kiểm tra thêm identity milled/rough từ Raw. Hệ số 1/10000 chỉ áp Attribute Milling Rate (.9999); raw_value/raw_unit bắt buộc giữ. Evidence được cố định trong notebook, Run All không cần Internet.

Nguồn đối chiếu: [USDA FAS, Vietnam annual 2010](https://apps.fas.usda.gov/newgainapi/api/report/downloadreportbyfilename?filename=Grain+and+Feed+Annual_Hanoi_Vietnam_4-5-2010.pdf), [USDA FAS, TU1012, bảng rice trang in 17 (PDF page 19)](https://apps.fas.usda.gov/newgainapi/api/Report/DownloadReportByFileName?fileName=Grain+and+Feed+Annual_Ankara_Turkey_03-01-2001.pdf), [ERS Rice Yearbook](https://www.ers.usda.gov/data-products/rice-yearbook). Ngày truy cập 2026-10-11. Tài liệu 2010 chứng minh quy ước lịch sử, không tự gán tháng bắt đầu cho mọi PSD year hay mọi export quote.


In [7]:
OFFICIAL_EVIDENCE={
    "milling_vietnam":"https://apps.fas.usda.gov/newgainapi/api/report/downloadreportbyfilename?filename=Grain+and+Feed+Annual_Hanoi_Vietnam_4-5-2010.pdf",
    "milling_formula":"https://apps.fas.usda.gov/newgainapi/api/Report/DownloadReportByFileName?fileName=Grain+and+Feed+Annual_Ankara_Turkey_03-01-2001.pdf",
    "yearbook":"https://www.ers.usda.gov/data-products/rice-yearbook"}
nso_metadata={}
metadata_path=RAW_ROOT/"nso"/"metadata.txt"
if "nso" in selected_sources and metadata_path.exists():
    for line in metadata_path.read_text(encoding="utf-8-sig").splitlines():
        match=re.match(r"(\d+)\.\s*(.*)",line)
        if match:nso_metadata["V06."+match.group(1)+".csv"]=match.group(2)
SEMANTICS={}
for ident,v in VIEWS.items():
    m=DATA[ident]["meta"];source=m["source"];name=Path(m["file"]).name
    if source=="faostat" and "Reporter Country Code" in v:
        info=dict(dataset="Detailed trade matrix",grain="reporter × partner × commodity/form × element × year × unit",
            time="Year labels; calendar/trade coverage requires release documentation",
            geography="reporter and partner source codes; country + region/world aggregates can coexist",
            measure="import/export quantity (t) and value (1000 USD); mirror flows are distinct",
            units="t → tonne; 1000 USD → USD ×1000 only for value elements; CPC prefix/decimal kept",
            additivity="only disjoint partner, product and geographic coverage; never add import/export mirrors or paddy-milled equivalent and components",
            unknowns="release identity, aggregate membership, calendar convention, bilateral mirror policy")
    elif source=="faostat" and "Months" in v:
        info=dict(dataset="Producer prices / indices",grain="area × item × element × reporting year × month-or-annual × unit",
            time="Monthly observations + Annual value, explicitly separate frequencies",
            geography="M49 lexical code and raw area label; LCU/SLC currency identities need source dictionary",
            measure="price (LCU/tonne, SLC/tonne, USD/tonne) vs development index 2014–2016=100",
            units="price basis is encoded in Element, not only Unit; index unit candidate index_points",
            additivity="prices and indices nonadditive; annual value not summed with monthly values",
            unknowns="LCU/SLC currency histories, month coverage expectations, aggregation weights and index treatment")
    elif source=="faostat" and "supply" in name:
        info=dict(dataset="Supply Utilization Accounts",grain="area × CPC item × element × year × unit",
            time="annual labels; source release calendar basis needs confirmation",
            geography="source M49 strings; aggregate area membership pending",
            measure="production, opening stocks, imports/exports, stock variation, residuals and uses",
            units="t → tonne with raw unit retained; rice vs rice milled remain separate items",
            additivity="flows only over disjoint coverage; stocks not additive over time; residuals/sign conventions require equation policy",
            unknowns="negative residual/stock policy, exact balance equation, processing basis and release identity")
    elif source=="faostat":
        info=dict(dataset="Crops/livestock production",grain="area × item × element × year × unit",
            time="annual reporting year; calendar/crop alignment pending",
            geography="M49 code + raw area, including any aggregates; no blind sum over all areas",
            measure="Area harvested, Yield, Production",
            units="ha → ha; kg/ha → kg/ha (or t/ha ×0.001); t → tonne, dimension-specific",
            additivity="area/production only across disjoint geography; yield nonadditive, weighted by matching harvested area",
            unknowns="harvested vs sown area compatibility, geographic aggregates, vintage and calendar basis")
    elif source=="nso":
        title=nso_metadata.get(name,"source metadata unavailable")
        national=DATA[ident].get("organization")=="statistic_year_measure_table"
        info=dict(dataset=title,grain=("statistic kind × reporting year × measure × season, national" if national else "raw geography label × year-column × file measure/season"),
            time="Reporting/crop year; So b? is provisional; no final status inferred for other rows",
            geography=("national table scope; source label retained" if national else "national/regions/provinces mixed, historical labels retained without modern code assignment"),
            measure=("physical quantities and previous-year=100 development indices in separate rows" if national else "measure/season from metadata.txt; identical schema does not mean identical measure"),
            units=("physical headers contain Nghìn ha / Nghìn t?n; index rows use % previous-year=100 despite physical column titles" if national else "units absent in matrix CSV; candidates thousand ha/thousand t and yield tạ/ha require official unit evidence"),
            additivity="physical quantities only over disjoint geographies/seasons; region totals overlap provinces; indices/yields nonadditive",
            unknowns="irreversible ? labels, historical boundaries and effective dates, matrix units, release status, hè thu + thu đông season coverage")
    elif source=="usda" and "Attribute" in v:
        info=dict(dataset="PSD Vietnam Rice, Milled (HTML export)",grain="commodity block × attribute × country × marketing/crop year",
            time="1960/1961 style periods retained; crop vs marketing/trade-year begin/end need evidence; TY not conflated with MY",
            geography="Country Vietnam; blank Commodity cells appear inside block",
            measure=f"{v.Attribute.nunique()} observed attributes including rough/milled production, Yield, MY and TY trade and Milling Rate",
            units="1000 HA ×1000 → ha; 1000 MT ×1000 → tonne, except Milling Rate (.9999) ×0.0001 → ratio; MT/HA → t/ha",
            additivity="stocks not additive over years; yields/ratios nonadditive; Total Supply/Distribution overlap components; MY/TY trade distinct",
            unknowns="commodity block propagation authorization, period start dates, release/vintage and forecast status")
    elif source=="usda":
        info=dict(dataset="Rice Yearbook export price quotes",grain="table × commodity × class × YEAR × reference period × location × statistic × unit",
            time="month labels and MARKETING aggregate; YEAR in August-start tables is not automatically calendar year",
            geography="country quote origin/location, not necessarily production geography",
            measure="FREIGHT ON BOARD PRICE, separate grades/classes and reporting periods",
            units="DOLLARS PER METRIC TON; USD/tonne candidate supported by ERS series context, retain raw label",
            additivity="prices nonadditive; MARKETING average not summed with monthly values; average weights unknown",
            unknowns="NA semantics, per-table YEAR-to-calendar conversion and partial year expected coverage")
    elif source=="world_bank":
        info=dict(dataset="Pink Sheet "+m["table"],grain="series/commodity specification × calendar month × statistic kind",
            time="YYYYMmm monthly; no annual dataset in this workbook",
            geography="origin/FOB/CIF specification in Description; commodity name is not country code",
            measure="nominal USD prices vs 2010=100 indices; every grade/series stays distinct",
            units="source unit row for prices; USD/mt → USD/tonne; indices remain index_points with base year",
            additivity="prices/indices nonadditive; annual aggregation needs weights/coverage policy",
            unknowns="ellipsis missing semantics, header update date inconsistent with data tail, vintage and description changes over time")
    else:
        info=dict(dataset="Domestic rice/paddy market quotes",grain="raw product × market × price kind × unit × currency × date × source (candidate)",
            time="month/day/year 12h timestamps, timezone absent; observed daily quote dates",
            geography="market labels; never fuzzy-map to province codes without approved dictionary",
            measure="price quotes; fresh/dry paddy, rice grades and local names remain distinct",
            units="VNĐ/Kg, Vnđ/Kg, VNĐ/kg, Đồng/kg + currency VNĐ; candidate VND/kg without value rescaling",
            additivity="prices nonadditive; average requires grain/product/market/price kind and weights",
            unknowns="zero-price meaning, conflicting quote duplicates, unit alias approval, product and source dictionaries")
    SEMANTICS[ident]=info
semantics_table=pd.DataFrame([{"id":ident,**info} for ident,info in SEMANTICS.items()])
show(semantics_table,50)

reference_rows=[]
trade_ids=[ident for ident,v in VIEWS.items() if "Reporter Country Code" in v]
lookup_specs=[("Reporter Country Code","ReporterCountries","Reporter Country Code"),
              ("Partner Country Code","PartnerCountries","Partner Country Code"),
              ("Item Code","ItemCodes","Item Code"),("Element Code","Elements","Element Code"),("Flag","Flags","Flag")]
for ident in trade_ids:
    v=VIEWS[ident]
    for column,marker,lookup_column in lookup_specs:
        matches=[e for key,e in DATA.items() if marker in Path(e["meta"]["file"]).name and e["meta"]["role"]=="reference_data"]
        if not matches:continue
        reference=matches[0]["raw"]
        unknown=~text_series(v[column]).isin(text_series(reference[lookup_column]))
        reference_rows.append({"id":ident,"field":column,"dictionary_file":matches[0]["meta"]["file"],
            "unknown_records":int(unknown.sum()),"unknown_values":str(v.loc[unknown,column].unique().tolist()[:8])})
reference_validation=pd.DataFrame(reference_rows)
show(reference_validation,30)


,id,dataset,grain,time,geography,measure,units,additivity,unknowns
0,faostat/price.csv,Producer prices / indices,area × item × element × reporting year × month-or-annual × unit,"Monthly observations + Annual value, explicitly separate frequencies",M49 lexical code and raw area label; LCU/SLC currency identities need source dictionary,"price (LCU/tonne, SLC/tonne, USD/tonne) vs development index 2014–2016=100","price basis is encoded in Element, not only Unit; index unit candidate index_points",prices and indices nonadditive; annual value not summed with monthly values,"LCU/SLC currency histories, month coverage expectations, aggregation weights and index trea..."
1,faostat/production_world.csv,Crops/livestock production,area × item × element × year × unit,annual reporting year; calendar/crop alignment pending,"M49 code + raw area, including any aggregates; no blind sum over all areas","Area harvested, Yield, Production","ha → ha; kg/ha → kg/ha (or t/ha ×0.001); t → tonne, dimension-specific","area/production only across disjoint geography; yield nonadditive, weighted by matching har...","harvested vs sown area compatibility, geographic aggregates, vintage and calendar basis"
2,faostat/supply_utilization.csv,Supply Utilization Accounts,area × CPC item × element × year × unit,annual labels; source release calendar basis needs confirmation,source M49 strings; aggregate area membership pending,"production, opening stocks, imports/exports, stock variation, residuals and uses",t → tonne with raw unit retained; rice vs rice milled remain separate items,flows only over disjoint coverage; stocks not additive over time; residuals/sign convention...,"negative residual/stock policy, exact balance equation, processing basis and release identity"
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Detailed trade matrix,reporter × partner × commodity/form × element × year × unit,Year labels; calendar/trade coverage requires release documentation,reporter and partner source codes; country + region/world aggregates can coexist,import/export quantity (t) and value (1000 USD); mirror flows are distinct,t → tonne; 1000 USD → USD ×1000 only for value elements; CPC prefix/decimal kept,"only disjoint partner, product and geographic coverage; never add import/export mirrors or ...","release identity, aggregate membership, calendar convention, bilateral mirror policy"
4,nso/V06.12.csv,Diện tích gieo trồng và sản lượng lúa cả năm,"statistic kind × reporting year × measure × season, national",Reporting/crop year; So b? is provisional; no final status inferred for other rows,national table scope; source label retained,physical quantities and previous-year=100 development indices in separate rows,physical headers contain Nghìn ha / Nghìn t?n; index rows use % previous-year=100 despite p...,physical quantities only over disjoint geographies/seasons; region totals overlap provinces...,"irreversible ? labels, historical boundaries and effective dates, matrix units, release sta..."
...,...,...,...,...,...,...,...,...,...
17,thitruongnongsan/price_luagao.xlsx::price_lua_gao,Domestic rice/paddy market quotes,raw product × market × price kind × unit × currency × date × source (candidate),"month/day/year 12h timestamps, timezone absent; observed daily quote dates",market labels; never fuzzy-map to province codes without approved dictionary,"price quotes; fresh/dry paddy, rice grades and local names remain distinct","VNĐ/Kg, Vnđ/Kg, VNĐ/kg, Đồng/kg + currency VNĐ; candidate VND/kg without value rescaling",prices nonadditive; average requires grain/product/market/price kind and weights,"zero-price meaning, conflicting quote duplicates, unit alias approval, product and source d..."
18,usda/Export-prices-Thailand-Vietnam-India-and-Pakistan.csv,Rice Yearbook export price quotes,table × commodity × class × YEAR × reference period × location × statistic × unit,month labels and MARKETING aggregate; YEAR in August-start tables is not automatica

,id,field,dictionary_file,unknown_records,unknown_values
0,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Reporter Country Code,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Repor...,0,[]
1,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Partner Country Code,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Partn...,0,[]
2,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Item Code,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_ItemC...,0,[]
3,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Element Code,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Eleme...,0,[]
4,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,Flag,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Flags...,0,[]


## STEP 7 — Cleaning Decision Table

Mỗi dòng là một quyết định cụ thể cho file/table, không phải tổng unique records của source. Counts có count_basis (records/cells/groups/headers); các issue có thể chồng lấp nên không cộng chúng để báo tổng records bị lỗi. evidence ghi tọa độ Raw; origin phân biệt RAW_PROFILE, RAW_SOURCE_METADATA và OFFICIAL_SOURCE.

Handling là hành động **đề xuất**. QUARANTINE chỉ dành cho parse failure, nonfinite, identifier không khớp dictionary hoặc key conflict khi đã có bằng chứng. Missing và outlier không tự quarantine. PENDING_REVIEW giữ Raw trong khi chờ nghiệp vụ. VERIFIED không phải “Silver cũ đã làm như vậy”.


In [8]:
ISSUES=[]
def raw_position(ident,index):
    meta=DATA[ident]["meta"]
    return int(index)+ (1 if meta["format"]=="xlsx" else 2)
def examples_from(ident,frame,mask=None,field=None):
    subset=frame.loc[mask] if mask is not None else frame
    result=[]
    for idx,row in subset.head(3).iterrows():
        position=int(row["raw_row"]) if "raw_row" in row and pd.notna(row["raw_row"]) else raw_position(ident,idx)
        column=str(row.get("raw_column",field or "record"))
        value=row.get(field) if field and field in row else row.get("raw_value",row.iloc[-1] if len(row) else "")
        result.append(f"row {position}, column {column}: {as_text(value)[:110]}")
    return "; ".join(result) or "No sampled values (scope-level decision)"
def add_issue(ident,field,issue,count,basis,examples,rule,canonical,handling,evidence,
              confidence="PROPOSED",approval="Data owner approval before Silver",test="Count/lineage reconciliation",origin="RAW_PROFILE"):
    if count==0:return
    ISSUES.append(dict(source=DATA[ident]["meta"]["source"],file=DATA[ident]["meta"]["file"],
        table=DATA[ident]["meta"]["table"],id=ident,field=field,issue_type=issue,affected_count=int(count),
        count_basis=basis,raw_examples=examples,proposed_rule=rule,canonical_type_unit=canonical,
        handling=handling,evidence=evidence,confidence=confidence,required_approval=approval,validation_test=test,origin=origin))

for ident,entry in DATA.items():
    d=entry["raw"];m=entry["meta"];source=m["source"]
    sp=schema_profile.loc[schema_profile.id.eq(ident)]
    if m["role"]=="observations":
        blanks=structural_mask(d)
        add_issue(ident,"all fields","structural_blank_row",int(blanks.sum()),"records",examples_from(ident,d,blanks),
            "Exclude only fully blank records from observation projection; preserve raw record lineage/count",
            "no observation", "EXCLUDE_STRUCTURAL","All cells blank/null/whitespace in raw row", "VERIFIED",
            "Review this explicit structural predicate","raw_rows = retained_rows + structural_blank_rows")
    for kind,col in [("duplicate_header","duplicate_header"),("blank_header","blank_header"),("normalized_name_collision","normalized_collision")]:
        bad=sp.loc[sp[col]]
        if len(bad):
            add_issue(ident,"header",kind,len(bad),"headers",str(bad[["column_position","raw_column"]].values.tolist()[:4]),
                "Preserve positional identity; approve unique canonical mapping before parsing named fields",
                "raw header + column ordinal","PENDING_REVIEW","Literal header layout","PROPOSED",
                "Approve canonical header mapping","No loss or collision in source-to-canonical column mapping")
    suspects=sp.loc[sp.encoding_suspect_count.gt(0)|sp.header_encoding_suspect]
    if len(suspects) and source=="nso":
        # Count distinct raw rows with suspect label and report header cells separately in evidence.
        bad= d.iloc[:,0].astype("string").str.contains(r"\?|�",na=False)
        add_issue(ident,str(d.columns[0]),"lossy_encoding_labels",int(bad.sum()),"records",
            examples_from(ident,d,bad,d.columns[0]),"Keep exact raw labels; request original Unicode export/approved exact dictionary; never fuzzy-map",
            "string raw_label + nullable reviewed code","PENDING_REVIEW",
            f"UTF-8 decode fails; '?' already in bytes; {int(sp.header_encoding_suspect.sum())} suspect headers. Decoder {m['encoding']} is candidate only",
            "UNKNOWN","Approve original source recovery and historical geography dictionary","No automatic '?' substitution or modern province assignment")
    identifiers=sp.loc[sp.leading_zero_count.gt(0)&sp.raw_column.str.contains("code|m49|cpc",case=False,regex=True)]
    if len(identifiers):
        affected=sum(int(r.leading_zero_count) for r in identifiers.itertuples())
        add_issue(ident,", ".join(identifiers.raw_column),"identifier_numeric_inference",affected,"cells",
            str(identifiers[["raw_column","examples"]].values.tolist()[:4]),"Retain lexical codes as string; never numeric cast before dictionary validation",
            "string codes","KEEP_RAW","Leading zeros present in raw lexical values","VERIFIED",
            "Accept lexical-code schema","Round-trip code strings preserves leading zeros and CPC fractions/prefixes")
    dup=duplicates.loc[duplicates.id.eq(ident)].iloc[0]
    if dup.exact_duplicate_excess:
        mask=d.duplicated(keep=False)
        add_issue(ident,"all fields","exact_duplicate_rows",int(dup.exact_duplicate_excess),"excess records",
            examples_from(ident,d,mask),"Propose lineage-preserving dedup only after repeat/release policy; retain all source references",
            "one observation + duplicate lineage","PENDING_REVIEW","Exact full-row equality, no hash-only test",
            "PROPOSED","Approve exact duplicate retention policy","Before/after difference equals approved duplicate excess")
    if source=="faostat" and m["role"]=="reference_data":
        spaced=[c for c in d.columns if str(c)!=str(c).strip()]
        if spaced:
            add_issue(ident,"header","header_whitespace",len(spaced),"headers",str(spaced),"Trim only canonical header alias; preserve raw header",
                "canonical string field name","NORMALIZE","Literal leading/trailing spaces in source header","VERIFIED",
                "Accept alias mapping","Canonical alias unique and raw name retained")

for ident,v in VIEWS.items():
    source=DATA[ident]["meta"]["source"];m=DATA[ident]["meta"]
    value_field=next((c for c in ("raw_value","Value","VALUE","Giá") if c in v),None)
    if not value_field:continue
    t=text_series(v[value_field]);n,commas=numeric_probe(v[value_field])
    blank=v[value_field].isna()|t.fillna("").str.strip().eq("")
    tokens=t.str.strip().isin(MISSING_TOKEN_CANDIDATES)
    if blank.any():
        verified=False
        if "Flag" in v and "Flag Description" in v:
            verified=bool((v.loc[blank,"Flag"].eq("M") & v.loc[blank,"Flag Description"].astype(str).str.contains("missing",case=False)).all())
        add_issue(ident,value_field,"blank_measure",int(blank.sum()),"observations",
            examples_from(ident,v,blank,value_field),"Preserve NULL measure with raw value/flag; never impute 0",
            "nullable decimal + missing_reason","NORMALIZE" if verified else "PENDING_REVIEW",
            "Blank raw measure"+("; Flag M description confirms missing" if verified else "; semantics not confirmed"),
            "VERIFIED" if verified else "PROPOSED","Approve retention/nullable schema","Missing count reconciles; no introduced numeric zero")
    if tokens.any():
        add_issue(ident,value_field,"missing_token_candidate",int(tokens.sum()),"observations",
            examples_from(ident,v,tokens,value_field),"Define source-specific token map with missing reason; retain original token; no zero imputation",
            "nullable decimal + raw token","PENDING_REVIEW",
            "Literal token distribution from full observation view; historical closure vs not available can differ",
            "PROPOSED","Approve source token semantics","Each approved token maps to NULL; unapproved tokens flagged")
    if commas.any():
        add_issue(ident,value_field,"thousands_separator",int(commas.sum()),"cells",
            examples_from(ident,v,commas,value_field),"Parse only strict comma-grouped thousands grammar; retain raw lexical value",
            "decimal","NORMALIZE","Values match grouped-thousands format; source HTML uses US number format",
            "VERIFIED","Accept numeric parser grammar","6,250 → 6250; reject ambiguous 1,25 and unrecognized tokens")
    failures=n.isna()&~blank&~tokens
    if failures.any():
        add_issue(ident,value_field,"numeric_parse_failure",int(failures.sum()),"observations",
            examples_from(ident,v,failures,value_field),"Quarantine lexical failures after approved token/locale parsing; retain raw/lineage",
            "nullable decimal + parse status","QUARANTINE","Nonblank, non-token values fail strict numeric projection",
            "PROPOSED","Approve parser and quarantine boundary","Every failure traceable to raw row/column; no coerced loss")
    nonfinite=n.notna()&~np.isfinite(n.astype(float))
    add_issue(ident,value_field,"nonfinite_numeric",int(nonfinite.sum()),"observations",examples_from(ident,v,nonfinite,value_field),
        "Quarantine infinities after parser policy","decimal","QUARANTINE","Numeric projection is nonfinite","VERIFIED",
        "Accept finite-number contract","isfinite for all non-NULL canonical numeric values")
    if n.lt(0).any():
        add_issue(ident,value_field,"negative_measure",int(n.lt(0).sum()),"observations",
            examples_from(ident,v,n.lt(0),value_field),"Keep negative values; flag by measure; residuals and stock variation sign policy pending",
            "signed decimal, source unit","PENDING_REVIEW","Negative values observed; no nonnegative business rule assumed",
            "UNKNOWN","Approve per-measure negative policy","Negative residual/stock values not automatically quarantined")
    nrow=numeric_profile.loc[numeric_profile.id.eq(ident)].iloc[0]
    if nrow.iqr_flagged:
        add_issue(ident,value_field,"iqr_outlier_candidate",int(nrow.iqr_flagged),"observations",
            examples_from(ident,range_examples[(ident,"iqr")],field=value_field),
            "Flag descriptive within-group IQR signals; review trend, geography and units; never auto-remove",
            "decimal + optional review flag","FLAG","IQR computed from all nonmissing values within "+nrow.iqr_grouping,
            "PROPOSED","Approve only business thresholds, if desired","Canonical retention count unchanged for IQR signals")
    ka=key_analysis.loc[key_analysis.id.eq(ident)]
    for row in ka.itertuples():
        if row.value_conflict_groups:
            mask=v.duplicated(KEYS[ident][row.candidate-1],keep=False)
            add_issue(ident,row.key,"business_key_value_conflict",int(row.value_conflict_groups),"key groups",
                examples_from(ident,v,mask,value_field),"Retain competing observations pending grain/release policy; quarantine only unresolved collisions of approved key; no last-wins",
                "observation + raw quote/source lineage","PENDING_REVIEW","Same candidate key has distinct raw measure values",
                "PROPOSED","Approve business key and conflict resolution","Approved-key conflicts either retained as distinct grain or explicitly quarantined")
    if source=="faostat":
        for c in [x for x in v.columns if re.search(r"M49|CPC",x)]:
            apostrophe=text_series(v[c]).str.startswith("'",na=False)
            if apostrophe.any():
                add_issue(ident,c,"code_apostrophe_prefix",int(apostrophe.sum()),"cells",
                    examples_from(ident,v,apostrophe,c),"Propose removing exactly one leading apostrophe in canonical code after local dictionary match; raw code unchanged",
                    "string canonical_code + raw_code","NORMALIZE","Same protective prefix occurs in local country/item lookup CSV",
                    "PROPOSED","Approve source-code parser including alpha/decimal CPC","Join success and lexical leading zeros preserved")
        if "Flag" in v:
            estimates=v["Flag"].isin(["E","I","X"])
            add_issue(ident,"Flag","source_estimate_provenance",int(estimates.sum()),"records",
                examples_from(ident,v,estimates,"Flag"),"Keep flags and descriptions as provenance; do not promote estimates to official/final",
                "raw flag + source status","KEEP_RAW","Flag labels/dictionary accompany raw observations","VERIFIED",
                "Accept source status retention","No loss of flags or notes through canonical mapping")
        if "Months" in v:
            annual=v.Months.eq("Annual value")
            add_issue(ident,"Months / Element","mixed_frequency_and_statistic",int(annual.sum()),"records",
                examples_from(ident,v,annual,"Months"),"Separate annual from monthly; price index from monetary price; interpret Unit together with Element",
                "frequency + statistic_kind + period_raw + nullable month","NORMALIZE",
                "Annual value and month labels coexist; Element explicitly says price index base 2014–2016",
                "VERIFIED","Approve separate canonical statistic/frequency fields","No fabricated month for annual or currency for index")
            indexmask=v.Element.astype(str).str.contains("Index",case=False)
            add_issue(ident,"Unit","index_blank_unit_is_structural",int((indexmask&v.Unit.eq("")).sum()),"records",
                examples_from(ident,v,indexmask&v.Unit.eq(""),"Unit"),"Preserve raw empty unit; set canonical index_points/base only for identified index Element",
                "index_points; base_period 2014–2016","NORMALIZE","Element explicitly identifies a base-100 index, not physical/currency measure",
                "VERIFIED","Accept index representation","Index rows never receive currency/tonne conversion")
        add_issue(ident,"release / geography / units","business_scope_metadata",len(v),"records",
            "Source code/category distributions in STEP 6","Approve geography aggregation membership, release identity and per-measure period/unit mapping",
            "versioned dimensions + period_basis + source_vintage","PENDING_REVIEW",SEMANTICS[ident]["unknowns"],
            "UNKNOWN","Approve source-specific contracts","Aggregation restricted to disjoint approved coverage")
    elif source=="nso":
        prov=v.provisional
        add_issue(ident,"period_raw","provisional_period",int(prov.sum()),"observations",
            examples_from(ident,v,prov,"period_raw"),"Retain reporting year candidate and explicit provisional status; no so_b → final",
            "integer year + provisional status + raw period","NORMALIZE","Literal So b? marker in raw period label",
            "VERIFIED","Accept provisional representation","All So b? observations tagged provisional; no unproved final status")
        if "statistic_raw" in v:
            indexmask=v.statistic_kind.eq("development_index")
            add_issue(ident,"statistic_raw / raw_column","index_vs_physical_units",int(indexmask.sum()),"observations",
                examples_from(ident,v,indexmask,"statistic_raw"),"Interpret unit by row statistic AND column measure; index rows keep %/previous-year=100, never thousand-unit scaling",
                "statistic_kind + base_period + percent_index","NORMALIZE","Raw statistic label explicitly contains previous-year =100 and %",
                "VERIFIED","Approve row-aware unit mapping","Index observations unaffected by physical scaling factors")
        else:
            add_issue(ident,"geography / units / season","historical_geography_and_missing_units",len(v),"observations",
                examples_from(ident,v,field="geography_raw"),"Keep exact historical labels and mixed levels; require effective-dated exact mapping and official matrix unit metadata",
                "nullable geography code/level + season + raw unit","PENDING_REVIEW",
                SEMANTICS[ident]["units"]+"; measure/season titles from nso/metadata.txt",
                "UNKNOWN","Approve historical dictionary, matrix units and seasonal scope",
                "No fuzzy mapping, no current province code backfill, no region/province double counting",origin="RAW_SOURCE_METADATA")
    elif source=="usda" and "Attribute" in v:
        commodity_blank=text_series(v.Commodity).fillna("").str.strip().eq("")
        add_issue(ident,"Commodity","grouped_commodity_blank",int(commodity_blank.sum()),"observations",
            examples_from(ident,v,commodity_blank,"Commodity"),"Propose filling Commodity only inside proven HTML commodity block, with boundaries and raw blank retained",
            "commodity candidate + raw Commodity","PENDING_REVIEW","First row Rice, Milled; subsequent attribute rows blank; HTML grouping",
            "PROPOSED","Approve bounded commodity propagation","No propagation across country/table/block boundary")
        milling=v.Attribute.eq("Milling Rate (.9999)")
        add_issue(ident,"Attribute / Unit Description / raw_value","milling_rate_encoding",int(milling.sum()),"cells",
            examples_from(ident,v,milling,"raw_value"),"Only Milling Rate (.9999): canonical_value = raw_numeric / 10000; always retain raw_value and raw_unit; canonical unit ratio",
            "decimal ratio (0..1) + raw unit/value","NORMALIZE",
            "Raw encoded values 6250/6270/6500/6600; official FAS Vietnam 2010: 62.5–62.7%; TU1012 formula ×10000. STEP 8 verifies milled/rough identity. "+OFFICIAL_EVIDENCE["milling_vietnam"]+" ; "+OFFICIAL_EVIDENCE["milling_formula"],
            "VERIFIED","Approve measure-scoped conversion before Silver",
            "6250 → 0.6250; reconstruction of rough×ratio matches milled within reporting precision; other attributes unchanged",origin="OFFICIAL_SOURCE")
        add_issue(ident,"period_raw / Attribute","marketing_crop_trade_year",len(v),"observations",
            examples_from(ident,v,field="period_raw"),"Keep year-span labels; distinguish MY and TY; approve period begin/end and vintage before calendar mapping",
            "period_raw + period_basis + nullable start/end dates","PENDING_REVIEW",
            "Year-span headers and TY attributes present; no begin-month/release fields in file",
            "UNKNOWN","Approve period and forecast/release identity","No date/year conversion from first four digits alone")
    elif source=="usda":
        marketing=v.REFERENCE_PERIOD_DESCRIPTION.eq("MARKETING")
        add_issue(ident,"YEAR / REFERENCE_PERIOD_DESCRIPTION","marketing_average_vs_month",int(marketing.sum()),"records",
            examples_from(ident,v,marketing,"REFERENCE_PERIOD_DESCRIPTION"),"Preserve MARKETING as annual marketing aggregate; other month-to-calendar conversions remain review-only",
            "reporting_year + reference_period + frequency + period_basis","PENDING_REVIEW",
            "Raw tables show 1985/86 or 1997/98 titles and MARKETING rows; ERS lists export quotes "+OFFICIAL_EVIDENCE["yearbook"],
            "PROPOSED","Approve per-table year/month convention and average weights","MARKETING never parsed as month; no aggregate/month double counting")
    elif source=="world_bank":
        add_issue(ident,"header / unit rows","multirow_header_and_units",len(WB_CONTEXT[ident]["structural_rows"]),"sheet rows",
            "Raw sheet rows "+str(WB_CONTEXT[ident]["structural_rows"][:12]),
            "Separate metadata/header/unit rows by explicit YYYY Mmm observation predicate; keep hierarchical index names and price unit row",
            "series_id + statistic_kind + month + raw unit + decimal","EXCLUDE_STRUCTURAL",
            "Monthly data rows match YYYY Mmm; actual header rows retained in raw grid","VERIFIED",
            "Approve sheet layout adapter","Observation + structural sheet rows reconcile; price/index separated")
        headertext="; ".join(as_text(x) for x in DATA[ident]["raw"].iloc[:4,0])
        yearmatch=re.search(r"Updated on .*?((?:19|20)\d{2})",headertext)
        data_year=pd.to_numeric(v.period_raw.astype(str).str[:4]).max()
        if yearmatch and data_year>=int(yearmatch.group(1)):
            # Compare actual full update date with latest observation month, not just year equality.
            update_text=next((as_text(x).replace("Updated on ","") for x in DATA[ident]["raw"].iloc[:4,0] if str(x).startswith("Updated on ")), "")
            update=pd.to_datetime(update_text,format="%B %d, %Y",errors="coerce")
            last=pd.to_datetime(v.period_raw,format="%YM%m").max()
            if pd.notna(update) and update<last:
                add_issue(ident,"sheet metadata","stale_update_header",1,"metadata fields",headertext,
                    "Flag source update header discrepancy; identify vintage by file hash, never treat header as reliable ingest timestamp",
                    "raw source note + content hash + reviewed vintage","FLAG",
                    f"Raw updated date {update.date()} precedes last monthly observation {last.date()}","VERIFIED",
                    "Confirm workbook release identity","All source date discrepancies retained/reported")
    else:
        zeros=n.eq(0)
        add_issue(ident,value_field,"zero_price",int(zeros.sum()),"records",
            examples_from(ident,v,zeros,value_field),"Preserve/flag zero; ask whether no-quote or true zero before NULL/quarantine rule",
            "decimal price + review flag","PENDING_REVIEW","Zero is a genuine numeric Raw value",
            "UNKNOWN","Approve zero-price business policy","No zero converted to NULL without approved source semantics")
        units=v["Đơn_vị_tính"].value_counts()
        if len(units)>1:
            add_issue(ident,"Đơn_vị_tính / Loại_tiền","unit_spelling_aliases",len(v),"records",
                str(units.to_dict()),"Propose exact alias dictionary for observed VND/kg spellings, no price rescaling; keep raw unit/currency",
                "VND/kg + raw_unit/currency","NORMALIZE","Raw currency VNĐ and per-kg labels agree; no tonne basis in observations",
                "PROPOSED","Approve four explicit observed aliases","Value unchanged; unit dictionary covers only approved exact strings")
        add_issue(ident,"Ngày","explicit_us_date_format",len(v),"records",
            examples_from(ident,v,field="Ngày"),"Parse %m/%d/%Y %I:%M:%S %p; retain raw timestamp; timezone unresolved, derive quote_date only after grain review",
            "local naive timestamp/date + raw date","NORMALIZE","Dates such as 7/31/2026 rule out day/month; full parse tested in STEP 5",
            "VERIFIED","Approve date grain and timezone policy","Full Raw date parse succeeds; no timezone invented")
        add_issue(ident,"Tên_mặt_hàng / Nguồn / Thị_trường","product_market_source_dictionary",len(v),"records",
            "Category distributions in STEP 5; source names mix collaborators, districts and Bán lẻ",
            "Retain product details fresh/dry/grade, raw markets and raw source; approve exact dictionaries, no fuzzy commodity merge",
            "raw strings + nullable reviewed dimension IDs","PENDING_REVIEW",SEMANTICS[ident]["unknowns"],
            "UNKNOWN","Approve product/geography/provenance dictionaries","No merging different processing basis, grades or markets")

for row in reference_validation.itertuples():
    add_issue(row.id,row.field,"unknown_reference_code",row.unknown_records,"records",row.unknown_values,
        "Flag/quarantine unmatched codes after confirming dictionary vintage; never drop silently",
        "raw code + reference_status","QUARANTINE","Unmatched to local source lookup "+row.dictionary_file,
        "PROPOSED","Approve reference vintage and allowed extras","Anti-join count reconciles; raw unmatched codes retained")
decisions=pd.DataFrame(ISSUES,columns=["source","file","table","id","field","issue_type","affected_count","count_basis","raw_examples","proposed_rule","canonical_type_unit","handling","evidence","confidence","required_approval","validation_test","origin"])
decisions.insert(0,"issue_id",[f"DQ-{i+1:03d}" for i in range(len(decisions))])
show(decisions[["issue_id","source","file","field","issue_type","affected_count","count_basis","handling","confidence"]],70)


,issue_id,source,file,field,issue_type,affected_count,count_basis,handling,confidence
0,DQ-001,faostat,faostat/price.csv,"Area Code (M49), Item Code (CPC)",identifier_numeric_inference,16951,cells,KEEP_RAW,VERIFIED
1,DQ-002,faostat,faostat/production_world.csv,"Area Code (M49), Item Code (CPC)",identifier_numeric_inference,25267,cells,KEEP_RAW,VERIFIED
2,DQ-003,faostat,faostat/supply_utilization.csv,"Area Code (M49), Item Code (CPC)",identifier_numeric_inference,23491,cells,KEEP_RAW,VERIFIED
3,DQ-004,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,"Reporter Country Code (M49), Partner Country Code (M49), Item Code (CPC)",identifier_numeric_inference,317518,cells,KEEP_RAW,VERIFIED
4,DQ-005,faostat,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Flags...,header,header_whitespace,1,headers,NORMALIZE,VERIFIED
...,...,...,...,...,...,...,...,...,...
65,DQ-066,nso,nso/V06.13.csv,period_raw,provisional_period,71,observations,NORMALIZE,VERIFIED
66,DQ-067,nso,nso/V06.13.csv,geography / units / season,historical_geography_and_missing_units,2130,observations,PENDING_REVIEW,UNKNOWN
67,DQ-068,nso,nso/V06.14.csv,raw_value,missing_token_candidate,44,observations,PENDING_REVIEW,PROPOSED
68,DQ-069,nso,nso/V06.14.csv,raw_value,iqr_outlier_candidate,48,observations,FLAG,PROPOSED


Displayed 70/132 rows; full table remains in memory.


## STEP 8 — Cross-source Checks

Chỉ kiểm tra khi grain, measure, unit và period có bằng chứng phù hợp. Các phép tính là diagnostic projections; không join để tạo Silver hoặc “sửa” nguồn. Production≈area×yield dùng đúng kg/ha→t factor và khoảng sai số suy từ precision Raw. Không gọi mọi chênh lệch là dữ liệu lỗi.

NSO so sánh label chính xác trong cùng file family, giữ national row và season scope; không cộng national+region+province. USDA ratio được tái dựng từ rough/milled của cùng marketing-year label. Trade unit value là USD/tonne diagnostic sau pairing quantity/value, không đồng nghĩa FOB quote hay producer price. USDA YEAR→calendar month còn pending nên không ép join với World Bank.


In [9]:
cross_rows=[];CROSS_DETAILS={}
def add_cross(relation,status,matched,findings,reason):
    cross_rows.append({"relation":relation,"status":status,"matched_observations":int(matched),"finding":findings,"compatibility_limits":reason})
def find_view(predicate):
    return [(ident,v) for ident,v in VIEWS.items() if predicate(ident,v)]
prod_matches=find_view(lambda ident,v:DATA[ident]["meta"]["source"]=="faostat" and Path(DATA[ident]["meta"]["file"]).name=="production_world.csv")
if prod_matches:
    ident,p=prod_matches[0];key=["Area Code (M49)","Item Code (CPC)","Year"]
    expected_units={"Area harvested":"ha","Yield":"kg/ha","Production":"t"}
    q=p.loc[p.Element.isin(expected_units)&p.apply(lambda r:expected_units.get(r.Element)==r.Unit,axis=1),key+["Element","Value"]].copy()
    if not q.duplicated(key+["Element"]).any():
        q["numeric"],_=numeric_probe(q.Value)
        # Resolution from literal Raw decimal precision; scientific notation is conservatively reviewed.
        places=text_series(q.Value).str.extract(r"\.(\d+)$",expand=False).str.len().fillna(0)
        q["resolution"]=10.**(-places.astype(int))
        wide=q.pivot(index=key,columns="Element",values="numeric").dropna()
        res=q.pivot(index=key,columns="Element",values="resolution").reindex(wide.index)
        wide["expected_t"]=wide["Area harvested"]*wide.Yield/1000.
        wide["difference_t"]=wide.Production-wide.expected_t
        wide["rounding_tolerance_t"]=(wide.Yield.abs()*res["Area harvested"]/2+
            wide["Area harvested"].abs()*res.Yield/2+res["Area harvested"]*res.Yield/4)/1000+res.Production/2
        wide["outside_precision_band"]=wide.difference_t.abs().gt(wide.rounding_tolerance_t+1e-6)
        CROSS_DETAILS["faostat_area_yield"]=wide.reset_index()
        add_cross("FAOSTAT production ↔ area × yield","DIAGNOSTIC",len(wide),
            f"{int(wide.outside_precision_band.sum())} outside lexical rounding band; max abs delta={wide.difference_t.abs().max():.6g} t",
            "Same area/item/year and explicit ha, kg/ha, t; discrepancies flagged for methodology/rounding review")
    else:add_cross("FAOSTAT production ↔ area × yield","INCOMPATIBLE",0,"Candidate key is not unique","No aggregation or winner selected")
    supply_matches=find_view(lambda ident,v:DATA[ident]["meta"]["source"]=="faostat" and "supply_utilization" in ident)
    if supply_matches:
        sid,s=supply_matches[0]
        left=p.loc[p.Element.eq("Production")&p.Unit.eq("t"),key+["Value"]].copy()
        right=s.loc[s.Element.eq("Production")&s.Unit.eq("t"),key+["Value"]].copy()
        if not left.duplicated(key).any() and not right.duplicated(key).any():
            joined=left.merge(right,on=key,how="inner",suffixes=("_production","_sua"),validate="one_to_one")
            a,_=numeric_probe(joined.Value_production);b,_=numeric_probe(joined.Value_sua)
            joined["difference_t"]=a-b
            valid=joined.difference_t.notna()
            CROSS_DETAILS["faostat_production_sua"]=joined
            add_cross("FAOSTAT Production ↔ SUA Production","DIAGNOSTIC",int(valid.sum()),
                f"{int(joined.loc[valid,'difference_t'].abs().gt(1e-6).sum())} different matched values",
                "Exact raw M49/CPC/year match, both t; release/calendar and processing definitions still require source confirmation")

psd_matches=find_view(lambda ident,v:"Attribute" in v and DATA[ident]["meta"]["source"]=="usda")
if psd_matches:
    ident,u=psd_matches[0]
    check=u.loc[u.Attribute.isin(["Production","Rough Production","Milling Rate (.9999)","Area Harvested","Yield"])].copy()
    check["numeric"],_=numeric_probe(check.raw_value)
    w=check.pivot(index="period_raw",columns="Attribute",values="numeric")
    ratio=w["Milling Rate (.9999)"]/10000.
    w["ratio_candidate"]=ratio
    w["milled_reconstruction_delta"]=w.Production-w["Rough Production"]*ratio
    w["rough_yield_delta"]=w.Yield-w["Rough Production"]/w["Area Harvested"]
    CROSS_DETAILS["usda_milling"]=w.reset_index()
    add_cross("USDA Milling Rate (.9999) ↔ milled / rough","VERIFIED_ENCODING",int(ratio.notna().sum()),
        f"ratio range {ratio.min():.4f}..{ratio.max():.4f}; max milled reconstruction delta {w.milled_reconstruction_delta.abs().max():.6g} thousand tonnes",
        "Official source evidence + same Raw period; 1/10000 applies only to Milling Rate, raw (1000 MT) label retained")
    add_cross("USDA Yield ↔ rough production / harvested area","DIAGNOSTIC",int(w.rough_yield_delta.notna().sum()),
        f"max abs delta {w.rough_yield_delta.abs().max():.6g} t/ha",
        "Observed rough-basis yield identity; never infer milled-basis yield from Commodity label alone")
    if prod_matches:
        add_cross("USDA PSD ↔ FAOSTAT production","INCOMPATIBLE",0,
            "USDA has rough and milled production; FAOSTAT has Rice and t/ha conventions",
            "USDA marketing/crop-year spans vs FAOSTAT reporting-year labels; no approved period/milling/geography mapping")

# NSO annual national table compared with national first label in geography matrix, exact raw lineage.
national_ids=[r.id for r in nso_groups.itertuples() if r.organization=="statistic_year_measure_table"] if len(nso_groups) else []
matrix_ids=[r.id for r in nso_groups.itertuples() if r.organization=="geography_year_matrix"] if len(nso_groups) else []
if national_ids and matrix_ids:
    nid=national_ids[0]; nv=VIEWS[nid]
    for mid in matrix_ids:
        title=nso_metadata.get(Path(DATA[mid]["meta"]["file"]).name,"")
        if "cả năm" not in title or not ("Diện tích" in title or "Sản lượng" in title):continue
        mv=VIEWS[mid];rawcol=DATA[nid]["raw"].columns[2 if "Diện tích" in title else 6]
        left=nv.loc[nv.statistic_kind.eq("physical_quantity")&nv.raw_column.eq(rawcol),["year_candidate","raw_value"]].copy()
        geo_label=as_text(DATA[mid]["raw"].iloc[0,0])
        right=mv.loc[mv.geography_raw.eq(geo_label),["year_candidate","raw_value"]].copy()
        joined=left.merge(right,on="year_candidate",how="inner",suffixes=("_national","_matrix"),validate="one_to_one")
        a,_=numeric_probe(joined.raw_value_national);b,_=numeric_probe(joined.raw_value_matrix)
        joined["difference_raw_units"]=a-b
        CROSS_DETAILS["nso_national_"+Path(mid).stem]=joined
        add_cross("NSO national ↔ "+Path(DATA[mid]["meta"]["file"]).name,"DIAGNOSTIC",len(joined),
            f"{int(joined.difference_raw_units.abs().gt(1e-6).sum())} different values in overlapping annual years",
            "Source metadata says annual area/production; exact first-row national label; equal values support but do not independently prove missing matrix unit")
    # Only identical labels/year/statistic within area/yield/production seasonal triples.
    title_groups={}
    for mid in matrix_ids:
        title=nso_metadata.get(Path(DATA[mid]["meta"]["file"]).name,"")
        suffix=re.sub(r"^(Diện tích(?: gieo trồng)?|Năng suất(?: gieo trồng)?|Sản lượng)\s+","",title)
        measure="area" if title.startswith("Diện tích") else "yield" if title.startswith("Năng suất") else "production" if title.startswith("Sản lượng") else "unknown"
        title_groups.setdefault(suffix,{})[measure]=mid
    for title,group in title_groups.items():
        if not {"area","yield","production"}<=set(group):continue
        pairs=[]
        for measure,mid in group.items():
            v=VIEWS[mid];part=v[["geography_raw","period_raw","raw_value"]].copy()
            part[measure],_=numeric_probe(part.raw_value);pairs.append(part.drop(columns="raw_value"))
        joined=pairs[0]
        for part in pairs[1:]:joined=joined.merge(part,on=["geography_raw","period_raw"],how="inner",validate="one_to_one")
        usable=joined.dropna(subset=["area","yield","production"])
        candidate=usable.area*usable["yield"]/10.
        delta=usable.production-candidate
        add_cross("NSO area/yield/production: "+title,"PENDING_UNIT_REVIEW",len(usable),
            f"hypothesis yield tạ/ha → production ≈ area × yield /10: median abs delta={delta.abs().median():.6g} raw production units",
            "Matrix unit metadata absent; factor /10 is a diagnostic hypothesis, not approved conversion; exact label joins only")

if trade_ids:
    ident=trade_ids[0];v=VIEWS[ident];key=["Reporter Country Code","Partner Country Code","Item Code","Year"]
    element_pairs=[("Import quantity","Import value"),("Export quantity","Export value")]
    for quantity,value in element_pairs:
        q=v.loc[v.Element.eq(quantity)&v.Unit.eq("t"),key+["Value"]].copy()
        a=v.loc[v.Element.eq(value)&v.Unit.eq("1000 USD"),key+["Value"]].copy()
        if q.duplicated(key).any() or a.duplicated(key).any():
            add_cross("Trade "+quantity+" ↔ "+value,"INCOMPATIBLE",0,"Nonunique pair key","No duplicate winner selected");continue
        joined=q.merge(a,on=key,suffixes=("_qty","_value"),validate="one_to_one")
        qty,_=numeric_probe(joined.Value_qty);val,_=numeric_probe(joined.Value_value)
        valid=qty.gt(0)&val.notna()
        joined["unit_value_usd_t"]=val*1000/qty.where(qty.gt(0))
        CROSS_DETAILS["trade_unit_value_"+quantity]=joined
        add_cross("Trade value/quantity ↔ rice price quotes","DIAGNOSTIC",int(valid.sum()),
            f"{quantity}: {int(qty.eq(0).sum())} zero-quantity pairs; median unit value={joined.unit_value_usd_t.median():.6g} USD/t",
            "Trade mix, partner, time basis and quantity grade differ from producer/FOB quotes; no automatic equality, no USD conversion for LCU/SLC")
wb_prices=find_view(lambda ident,v:DATA[ident]["meta"]["source"]=="world_bank" and DATA[ident]["meta"]["table"]=="Monthly Prices")
export_prices=find_view(lambda ident,v:"REFERENCE_PERIOD_DESCRIPTION" in v)
if wb_prices and export_prices:
    add_cross("World Bank rice quotes ↔ USDA Yearbook export quotes","PENDING_PERIOD_REVIEW",0,
        "Both quote prices can have USD/tonne basis, but grade/FOB and quote sample differ",
        "World Bank calendar month vs USDA reporting YEAR/month/MARKETING; approved per-table calendar mapping missing")
cross_checks=pd.DataFrame(cross_rows,columns=["relation","status","matched_observations","finding","compatibility_limits"])
show(cross_checks,50)
for name,table in CROSS_DETAILS.items():
    if name.startswith("trade"):
        show(table.sort_values("unit_value_usd_t",ascending=False).head(4),4)
    else:
        show(table.head(5),5)


,relation,status,matched_observations,finding,compatibility_limits
0,FAOSTAT production ↔ area × yield,DIAGNOSTIC,7311,0 outside lexical rounding band; max abs delta=2147.1 t,"Same area/item/year and explicit ha, kg/ha, t; discrepancies flagged for methodology/roundi..."
1,FAOSTAT Production ↔ SUA Production,DIAGNOSTIC,1553,343 different matched values,"Exact raw M49/CPC/year match, both t; release/calendar and processing definitions still req..."
2,USDA Milling Rate (.9999) ↔ milled / rough,VERIFIED_ENCODING,66,ratio range 0.6250..0.6600; max milled reconstruction delta 0.32 thousand tonnes,"Official source evidence + same Raw period; 1/10000 applies only to Milling Rate, raw (1000..."
3,USDA Yield ↔ rough production / harvested area,DIAGNOSTIC,66,max abs delta 0.00542614 t/ha,Observed rough-basis yield identity; never infer milled-basis yield from Commodity label alone
4,USDA PSD ↔ FAOSTAT production,INCOMPATIBLE,0,USDA has rough and milled production; FAOSTAT has Rice and t/ha conventions,USDA marketing/crop-year spans vs FAOSTAT reporting-year labels; no approved period/milling...
5,NSO national ↔ V06.13.csv,DIAGNOSTIC,30,3 different values in overlapping annual years,Source metadata says annual area/production; exact first-row national label; equal values s...
6,NSO national ↔ V06.15.csv,DIAGNOSTIC,30,3 different values in overlapping annual years,Source metadata says annual area/production; exact first-row national label; equal values s...
7,NSO area/yield/production: lúa cả năm phân theo địa phương,PENDING_UNIT_REVIEW,2077,hypothesis yield tạ/ha → production ≈ area × yield /10: median abs delta=0.2 raw production...,"Matrix unit metadata absent; factor /10 is a diagnostic hypothesis, not approved conversion..."
8,NSO area/yield/production: lúa đông xuân phân theo địa phương,PENDING_UNIT_REVIEW,2073,hypothesis yield tạ/ha → production ≈ area × yield /10: median abs delta=0.084 raw producti...,"Matrix unit metadata absent; factor /10 is a diagnostic hypothesis, not approved conversion..."
9,NSO area/yield/production: lúa hè thu và thu đông phân theo địa phương,PENDING_UNIT_REVIEW,1056,hypothesis yield tạ/ha → production ≈ area × yield /10: median abs delta=0.104 raw producti...,"Matrix unit metadata absent; factor /10 is a diagnostic hypothesis, not approved conversion..."


Element,Area Code (M49),Item Code (CPC),Year,Area harvested,Production,Yield,expected_t,difference_t,rounding_tolerance_t,outside_precision_band
0,004,0113,1961,210000.0,319000.0,1519.0,318990.0,10.0,106.25975,False
1,004,0113,1962,210000.0,319000.0,1519.0,318990.0,10.0,106.25975,False
2,004,0113,1963,210000.0,319000.0,1519.0,318990.0,10.0,106.25975,False
3,004,0113,1964,220000.0,380000.0,1727.3,380006.0,-6.0,12.363675,False
4,004,0113,1965,220000.0,380000.0,1727.3,380006.0,-6.0,12.363675,False


,Area Code (M49),Item Code (CPC),Year,Value_production,Value_sua,difference_t
0,004,0113,2010,672000,672000,0.0
1,004,0113,2011,672000,672000,0.0
2,004,0113,2012,703000,500000,203000.0
3,004,0113,2013,764000,512094,251906.0
4,004,0113,2014,801500,537000,264500.0


Attribute,period_raw,Area Harvested,Milling Rate (.9999),Production,Rough Production,Yield,ratio_candidate,milled_reconstruction_delta,rough_yield_delta
0,1960/1961,4602.0,6500.0,5959.0,9168.0,1.99,0.65,-0.2,-0.002177
1,1961/1962,4763.0,6500.0,6018.0,9258.0,1.94,0.65,0.3,-0.003733
2,1962/1963,4889.0,6500.0,6335.0,9746.0,1.99,0.65,0.1,-0.003455
3,1963/1964,4902.0,6500.0,6255.0,9623.0,1.96,0.65,0.05,-0.003076
4,1964/1965,4983.0,6500.0,6303.0,9697.0,1.95,0.65,-0.05,0.003984


,year_candidate,raw_value_national,raw_value_matrix,difference_raw_units
0,1995,6765.6,6765.6,0.0
1,1996,7003.8,7003.8,0.0
2,1997,7099.7,7099.7,0.0
3,1998,7362.7,7362.7,0.0
4,1999,7653.6,7653.6,0.0


,year_candidate,raw_value_national,raw_value_matrix,difference_raw_units
0,1995,24963.7,24963.7,0.0
1,1996,26396.7,26192.0,204.7
2,1997,27523.9,27288.7,235.2
3,1998,29145.5,28919.3,226.2
4,1999,31393.8,31393.8,0.0


,Reporter Country Code,Partner Country Code,Item Code,Year,Value_qty,Value_value,unit_value_usd_t
71065,48,44,30,2022,0.01,88.0,8800000.0
71063,48,44,30,2021,0.01,80.0,8000000.0
131865,202,96,32,2015,0.15,1188.0,7920000.0
70947,48,44,27,2022,0.02,88.0,4400000.0


,Reporter Country Code,Partner Country Code,Item Code,Year,Value_qty,Value_value,unit_value_usd_t
105597,215,229,30,2018,0.01,110.0,11000000.0
105528,215,229,32,2018,0.01,110.0,11000000.0
41445,44,48,30,2021,0.01,80.0,8000000.0
185574,110,210,29,2017,0.01,54.0,5400000.0


## STEP 9 — EDA Summary & validation

Tổng issue dưới đây đếm dòng decision theo source, không cộng chồng lấp record/cell counts. VERIFIED có evidence để thiết kế; triển khai Silver vẫn phải review. PENDING_REVIEW và QUARANTINE candidates liệt kê riêng.

Kiểm tra reproducibility gồm hash toàn bộ file Raw/tài liệu cũ trước–sau, inventory coverage, observation lineage, numeric accounting, missing classes, key conflicts và trạng thái quyết định. Run All thành công chỉ nghĩa notebook thực thi và các invariant kiểm tra đạt; không đồng nghĩa Raw sạch hoặc mọi quy tắc nghiệp vụ được duyệt.


In [10]:
issue_summary=decisions.groupby("source").agg(decision_rows=("issue_id","size"),
    verified=("confidence",lambda s:int(s.eq("VERIFIED").sum())),
    pending_review=("handling",lambda s:int(s.eq("PENDING_REVIEW").sum())),
    quarantine_candidates=("handling",lambda s:int(s.eq("QUARANTINE").sum()))).reset_index()
show(issue_summary,50)
verified_rules=decisions.loc[decisions.confidence.eq("VERIFIED")]
pending_rules=decisions.loc[decisions.handling.eq("PENDING_REVIEW")]
quarantine_candidates=decisions.loc[decisions.handling.eq("QUARANTINE")]
show(verified_rules[["issue_id","file","issue_type","proposed_rule","validation_test"]],35)
show(pending_rules[["issue_id","file","issue_type","required_approval"]],35)
show(quarantine_candidates[["issue_id","file","issue_type","affected_count","count_basis"]],25)
print("Observed anomalies/flags are not a global DQ FAIL. No proposed rule applied to a production pipeline.")
print("Silver recommendation: small per-layout adapters; raw value/unit/period/label + file hash + row/column lineage; exact reviewed dictionaries; measure-scoped conversion; explicit missing and version policies.")
checks=[]
def check(name,condition,detail):
    checks.append({"check":name,"status":"PASS" if bool(condition) else "FAIL","detail":detail})
check("Full input coverage",set(p.relative_to(RAW_ROOT).as_posix() for p in selected_paths)==set(inventory.file),
      "Every selected data file loaded, including reference CSVs and all workbook sheets")
check("No sampling",inventory.scope.eq("FULL_DATA").all(),"All reported counts are full-data")
if "nso" in selected_sources:
    expected={p.relative_to(RAW_ROOT).as_posix() for p in (RAW_ROOT/"nso").glob("V06.*.csv")}
    actual=set(nso_groups.id) if len(nso_groups) else set()
    check("NSO complete grouping",actual==expected,"Every V06.* read even with NSO file filter")
check("CSV header/body width",all(e["raw"].shape[1]==len(e["meta"]["headers"]) for e in DATA.values()),"Positional headers preserved")
check("Numeric accounting",all(r.numeric_valid+r.missing_candidates+r.parse_failures==r.observations for r in numeric_profile.itertuples()),
      "Valid + missing candidates + actual parse failures = observation count")
check("Missing classes do not overlap",all(r.unresolved_missing_candidates<=r.denominator_cells for r in missing_profile.itertuples()),
      "NULL/empty/whitespace/token classes accounted without double-counting")
check("Decision evidence",decisions.evidence.ne("").all() and decisions.validation_test.ne("").all(),
      "Every decision has evidence and proposed validation")
check("Decision enums",set(decisions.handling)<=set("NORMALIZE KEEP_RAW FLAG QUARANTINE EXCLUDE_STRUCTURAL PENDING_REVIEW".split())
      and set(decisions.confidence)<=set("VERIFIED PROPOSED UNKNOWN".split()),"Only allowed handling/confidence")
check("Raw and old artifacts unchanged",all(sha256_file(RAW_ROOT/rel)==digest for rel,digest in initial_hashes.items()),
      f"{len(initial_hashes)} original files hash-checked")
for ident,v in VIEWS.items():
    if "raw_row" in v:
        check("Lineage: "+ident,v.raw_row.notna().all(),"Every projected observation points to original row and column")
if psd_matches:
    w=CROSS_DETAILS["usda_milling"]
    # Raw 1000 MT values reported to integer precision; half-unit quantization from both quantities.
    ratio=w.ratio_candidate
    tol=.5+.5*ratio.abs()+1e-6
    check("Milling factor identity",w.milled_reconstruction_delta.abs().le(tol).all() and ratio.between(0,1).all(),
          "Milled ≈ rough×encoded/10000 within source integer reporting precision")
validation_results=pd.DataFrame(checks)
show(validation_results,70)
assert validation_results.status.eq("PASS").all(), "See FAIL rows; do not claim Run All PASS"
display(Markdown(f"**Execution invariants PASS:** {len(validation_results)} checks; {inventory.file.nunique()} files / {len(inventory)} tables; {len(decisions)} decision rows. Business approvals remain open."))


,source,decision_rows,verified,pending_review,quarantine_candidates
0,faostat,28,16,5,0
1,nso,77,27,37,0
2,thitruongnongsan,8,1,5,0
3,usda,8,2,5,0
4,world_bank,11,5,4,0


,issue_id,file,issue_type,proposed_rule,validation_test
0,DQ-001,faostat/price.csv,identifier_numeric_inference,Retain lexical codes as string; never numeric cast before dictionary validation,Round-trip code strings preserves leading zeros and CPC fractions/prefixes
1,DQ-002,faostat/production_world.csv,identifier_numeric_inference,Retain lexical codes as string; never numeric cast before dictionary validation,Round-trip code strings preserves leading zeros and CPC fractions/prefixes
2,DQ-003,faostat/supply_utilization.csv,identifier_numeric_inference,Retain lexical codes as string; never numeric cast before dictionary validation,Round-trip code strings preserves leading zeros and CPC fractions/prefixes
3,DQ-004,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/rice_trade_data.csv,identifier_numeric_inference,Retain lexical codes as string; never numeric cast before dictionary validation,Round-trip code strings preserves leading zeros and CPC fractions/prefixes
4,DQ-005,faostat/Trade_DetailedTradeMatrix_E_All_Data_(Normalized)/Trade_DetailedTradeMatrix_E_Flags...,header_whitespace,Trim only canonical header alias; preserve raw header,Canonical alias unique and raw name retained
...,...,...,...,...,...
61,DQ-062,nso/V06.12.csv,provisional_period,Retain reporting year candidate and explicit provisional status; no so_b → final,All So b? observations tagged provisional; no unproved final status
62,DQ-063,nso/V06.12.csv,index_vs_physical_units,"Interpret unit by row statistic AND column measure; index rows keep %/previous-year=100, ne...",Index observations unaffected by physical scaling factors
65,DQ-066,nso/V06.13.csv,provisional_period,Retain reporting year candidate and explicit provisional status; no so_b → final,All So b? observations tagged provisional; no unproved final status
69,DQ-070,nso/V06.14.csv,provisional_period,Retain reporting year candidate and explicit provisional status; no so_b → final,All So b? observations tagged provisional; no unproved final status


Displayed 35/51 rows; full table remains in memory.


,issue_id,file,issue_type,required_approval
9,DQ-010,nso/V06.12.csv,lossy_encoding_labels,Approve original source recovery and historical geography dictionary
11,DQ-012,nso/V06.13.csv,lossy_encoding_labels,Approve original source recovery and historical geography dictionary
13,DQ-014,nso/V06.14.csv,lossy_encoding_labels,Approve original source recovery and historical geography dictionary
15,DQ-016,nso/V06.15.csv,lossy_encoding_labels,Approve original source recovery and historical geography dictionary
17,DQ-018,nso/V06.16.csv,lossy_encoding_labels,Approve original source recovery and historical geography dictionary
...,...,...,...,...
78,DQ-079,nso/V06.16.csv,historical_geography_and_missing_units,"Approve historical dictionary, matrix units and seasonal scope"
79,DQ-080,nso/V06.17.csv,missing_token_candidate,Approve source token semantics
82,DQ-083,nso/V06.17.csv,historical_geography_and_missing_units,"Approve historical dictionary, matrix units and seasonal scope"
83,DQ-084,nso/V06.18.csv,missing_token_candidate,Approve source token semantics


Displayed 35/56 rows; full table remains in memory.


,issue_id,file,issue_type,affected_count,count_basis


Observed anomalies/flags are not a global DQ FAIL. No proposed rule applied to a production pipeline.
Silver recommendation: small per-layout adapters; raw value/unit/period/label + file hash + row/column lineage; exact reviewed dictionaries; measure-scoped conversion; explicit missing and version policies.


,check,status,detail
0,Full input coverage,PASS,"Every selected data file loaded, including reference CSVs and all workbook sheets"
1,No sampling,PASS,All reported counts are full-data
2,NSO complete grouping,PASS,Every V06.* read even with NSO file filter
3,CSV header/body width,PASS,Positional headers preserved
4,Numeric accounting,PASS,Valid + missing candidates + actual parse failures = observation count
...,...,...,...
21,Lineage: nso/V06.24.csv,PASS,Every projected observation points to original row and column
22,Lineage: usda/usda.xls::html_table_1,PASS,Every projected observation points to original row and column
23,Lineage: world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Prices,PASS,Every projected observation points to original row and column
24,Lineage: world_bank/CMO-Historical-Data-Monthly.xlsx::Monthly Indices,PASS,Every projected observation points to original row and column


**Execution invariants PASS:** 26 checks; 26 files / 30 tables; 132 decision rows. Business approvals remain open.

## Cleaning Markdown exporter — optional, reviewable output

Các specification được dựng trong bộ nhớ từ kết quả STEP 1–9 và có đủ 12 phần. Workbook nhiều sheet chung một spec, lookup trade đi chung spec trade, NSO một spec cho mỗi nhóm schema thực tế. Không tạo helper script/JSON/CSV/log.

WRITE_SPECS=False để Run All chỉ đọc Raw. Đặt True ở STEP 0 để xuất Markdown sau khi invariants đạt. Nếu file cùng tên đã có nội dung khác, exporter dùng suffix _v2 rồi _v2_2; nội dung gốc luôn được bảo toàn. Nếu nội dung giống hệt thì không ghi lại. Chạy subset chỉ tạo specification có đánh dấu phạm vi subset; không giả là full-source coverage.


In [11]:
def md_table(frame):
    if frame.empty:return "_Không có bản ghi trong phạm vi phân tích._"
    def escape(v):
        if isinstance(v,float) and np.isnan(v):return ""
        return str(v).replace("|","\\|").replace("\n","<br>").replace("\r","")
    headers=[escape(c) for c in frame.columns]
    lines=["| "+" | ".join(headers)+" |","| "+" | ".join("---" for _ in headers)+" |"]
    lines += ["| "+" | ".join(escape(v) for v in row)+" |" for row in frame.itertuples(index=False,name=None)]
    return "\n".join(lines)

COMMON_SCHEMA=[
    ("source_file","string","Raw relative path"),
    ("source_sheet_or_table","string nullable","Workbook sheet/HTML table when present"),
    ("source_content_sha256","string","Snapshot identity, not business release"),
    ("raw_row","long","CSV record ordinal or Excel physical row"),
    ("raw_column","string","Original column/ordinal for wide projections"),
    ("raw_value","string nullable","Keep source lexical value/native-cell provenance"),
    ("raw_unit","string nullable","Keep exact Raw unit, even anomalous/blank"),
    ("canonical_value","decimal, precision pending review","Never silently coerce invalid numeric"),
    ("canonical_unit","string nullable","Only approved measure-scoped mapping"),
    ("missing_reason","string nullable","NULL/blank/token/structural semantics kept distinct"),
    ("dq_status","string","Retained/flagged/quarantined, proposed statuses"),
    ("source_vintage","string nullable","Business release identity still pending when absent")]
def canonical_schema(source,ident):
    v=VIEWS.get(ident,pd.DataFrame())
    if source=="faostat" and "Reporter Country Code" in v:
        fields=[("reporter_code","string","source code system + raw M49"),("partner_code","string","country or aggregate"),
                ("item_code","string","preserve raw CPC/form"),("element_code","string","quantity/value and direction"),
                ("reporting_year","integer","period basis pending"),("statistic_kind","string","quantity or trade value"),
                ("flag","string","raw source estimate/status")]
    elif source=="faostat":
        fields=[("area_code_m49","string","retain leading zeros"),("item_code_cpc","string","retain decimal/alpha codes"),
                ("element_code","string","measure semantics"),("reporting_year","integer","year basis explicit"),
                ("month","integer nullable","NULL for annual"),("frequency","string","monthly/annual"),
                ("statistic_kind","string","quantity/yield/price/index"),("currency","string nullable","LCU/SLC identity pending"),
                ("price_basis","string nullable","per tonne from Element"),("index_base_period","string nullable","from Element"),
                ("flag","string","source status"),("raw_note","string nullable","retain source notes")]
    elif source=="nso":
        fields=[("geography_raw","string","exact lossy/historical label"),("geography_code","string nullable","approved effective-dated map only"),
                ("geography_level","string nullable","national/region/province awaiting mapping"),("boundary_version","string nullable","historical coverage"),
                ("reporting_year","integer","parsed raw year label"),("period_raw","string","retain provisional label"),
                ("release_status","string","provisional where explicit; otherwise unconfirmed"),
                ("measure","string","source metadata/header"),("season","string","all-year/đông xuân/hè thu+thu đông/mùa, exact scope"),
                ("statistic_kind","string","physical quantity vs development index"),("index_base","string nullable","previous year=100")]
    elif source=="usda" and "Attribute" in v:
        fields=[("commodity_raw","string nullable","blank block labels retained"),("commodity","string nullable","bounded propagation pending"),
                ("country_raw","string","Vietnam source label"),("attribute_raw","string","MY/TY/milling/rough/yield kept"),
                ("period_raw","string","YYYY/YYYY"),("period_basis","string","marketing/crop/trade pending attribute policy"),
                ("period_start","date nullable","no fabricated start date"),("period_end","date nullable","needs source evidence"),
                ("statistic_kind","string","quantity/area/yield/ratio"),("processing_basis","string nullable","rough/milled"),
                ("conversion_factor","decimal nullable","0.0001 only Milling Rate; ×1000 only physical thousand unit")]
    elif source=="usda":
        fields=[("table_number","string","retain table identity"),("commodity_raw","string","RICE"),
                ("class_raw","string","grade/product specification"),("location_raw","string","quote origin"),
                ("reporting_year","integer","Raw YEAR"),("reference_period_raw","string","month/MARKETING"),
                ("frequency","string","month/marketing average"),("calendar_month","date nullable","requires approved YEAR convention"),
                ("statistic_kind","string","FOB quote"),("currency","string","USD candidate"),("price_basis","string","metric tonne")]
    elif source=="world_bank":
        fields=[("series_raw","string","exact header/hierarchical index label"),("series_id","string","approved stable specification"),
                ("month","date","calendar month from YYYYMmm"),("statistic_kind","string","price/index"),
                ("currency","string nullable","nominal USD for monetary series"),("price_basis","string nullable","source unit row"),
                ("index_base","string nullable","2010=100 for indices"),
                ("series_description","string nullable","Description sheet, dated definition changes retained")]
    else:
        fields=[("product_raw","string","variety/freshness/grade retained"),("product_id","string nullable","approved exact dictionary"),
                ("market_raw","string","source location"),("price_kind_raw","string","quote context"),
                ("currency","string","VND candidate"),("price_basis","string","kg candidate, no rescaling"),
                ("quote_datetime_raw","string","source timestamp"),("quote_date","date","date grain approval pending"),
                ("quote_timezone","string nullable","absent in Raw"),("quote_source_raw","string","source/provenance context")]
    return pd.DataFrame(COMMON_SCHEMA+fields,columns=["canonical_field","proposed_type","meaning/evidence_limit"])

SPEC_PLAN={}
for ident,entry in DATA.items():
    source=entry["meta"]["source"];name=Path(entry["meta"]["file"]).name
    if source=="nso":
        group=nso_groups.loc[nso_groups.id.eq(ident),"group"].iloc[0]
        path="nso/cleaning_"+group+".md"
    elif source=="faostat":
        if entry["meta"]["role"]=="reference_data" or "rice_trade_data" in name:path="faostat/cleaning_trade.md"
        elif "production_world" in name:path="faostat/cleaning_production.md"
        elif name=="price.csv":path="faostat/cleaning_monthly_price.md"
        elif "supply" in name:path="faostat/cleaning_supply_utilization.md"
        else:path="faostat/cleaning_"+candidate_name(Path(name).stem)+".md"
    elif source=="usda":
        path="usda/cleaning_psd.md" if entry["meta"]["format"].startswith("html") or name=="usda.xls" else "usda/cleaning_rice_yearbook.md"
    elif source=="world_bank":path="world_bank/cleaning_pinksheet.md"
    else:path="thitruongnongsan/cleaning_prices.md"
    SPEC_PLAN.setdefault(path,[]).append(ident)

def make_spec(path,ids):
    source=path.split("/")[0]
    obs_ids=[ident for ident in ids if ident in VIEWS]
    representative=obs_ids[0] if obs_ids else ids[0]
    info=SEMANTICS.get(representative,{})
    inv=inventory.loc[inventory.id.isin(ids)]
    local_dec=decisions.loc[decisions.id.isin(ids)]
    local_num=numeric_profile.loc[numeric_profile.id.isin(ids)]
    local_period=period_profile.loc[period_profile.id.isin(ids)]
    local_miss=observation_missing_profile.loc[observation_missing_profile.id.isin(ids)]
    local_keys=key_analysis.loc[key_analysis.id.isin(ids)]
    local_dup=duplicates.loc[duplicates.id.isin(ids)]
    fingerprint_table=inv[["file","table","role","format","encoding","bytes","rows","columns","physical_rows","schema_fingerprint","content_sha256"]]
    membership=pd.DataFrame([{"file":DATA[ident]["meta"]["file"],"measure/season":SEMANTICS.get(ident,{}).get("dataset","lookup/metadata"),
        "observation_rows":len(VIEWS[ident]) if ident in VIEWS else 0} for ident in ids])
    sections=[
        f"# Cleaning specification — {Path(path).stem}\n\nReview snapshot: {AS_OF_DATE.date()}. Generated from EDA_v2.ipynb; counts FULL_DATA for selected inputs. "+
        f"Source/file selection: SOURCE_FILTER={SOURCE_FILTER!r}, FILE_FILTER={FILE_FILTER!r}. "+
        "Đây là thiết kế để review, chưa triển khai Silver. VERIFIED là đủ evidence cho đề xuất, không thay phê duyệt.\n\n"+
        "Counts records/cells/key groups/headers khác nhau và có overlap; không cộng affected_count để báo unique bad records.",
        "## 1. Dataset/source overview\n\n"+str(info.get("dataset","Source lookup"))+"\n\n"+
        "Nguồn kết luận: RAW_PROFILE là quan sát trực tiếp; RAW_SOURCE_METADATA là source metadata đi cùng Raw; OFFICIAL_SOURCE là tài liệu USDA được đối chiếu sau profiling. "+
        "Không sử dụng Silver hoặc eda_*.md cũ để xác nhận quy tắc.\n\n"+md_table(membership),
        "## 2. Input files và schema fingerprint\n\n"+md_table(fingerprint_table)+"\n\n"+
        "rows là Raw parser records (Excel direct table đã tách header row); physical_rows là sheet grid size/CSV record count gồm header; "+
        "schema fingerprint SHA256 rút gọn 16 ký tự của literal headers + format; full content hash cố định snapshot. "+
        "CSV cp1252/latin-1 là decoder candidate, chưa chứng minh encoding gốc. Lookup CSV và workbook metadata được ghi rõ role.",
        "## 3. Data grain và business semantics\n\n"+
        "\n".join("- **"+key+"**: "+str(info.get(key,"needs review")) for key in ["grain","time","geography","measure","units","additivity"])+
        "\n\n"+("### NSO schema membership and coverage\n\n"+md_table(nso_groups.loc[nso_groups.id.isin(ids)]) if source=="nso" else ""),
        "## 4. Raw profiling findings\n\n### Numeric values\n\n"+md_table(local_num)+
        "\n\n### Date/year validation\n\n"+md_table(local_period)+
        "\n\n### Raw exact duplicates\n\n"+md_table(local_dup)+
        "\n\nIQR flags là exploratory signals theo grouping ở bảng, không kết luận lỗi. Precision là số decimal digits quan sát, chưa phải canonical decimal scale. "+
        "Negative values giữ nguyên chờ policy theo measure. Zero không mặc định missing.",
        "## 5. Missing / duplicate / type / unit / geography issues\n\n"+
        md_table(local_miss)+"\n\n"+
        "NULL, empty, whitespace, tokens và zero tách riêng. Full per-column Raw counts và native dtypes nằm trong STEP 2–3; "+
        "bảng dưới ghi counts quan sát theo file và cột cần xử lý, gồm structural blank/header cells và leading-zero risk:\n\n"+
        md_table(file_missing_summary.loc[file_missing_summary.id.isin(ids)])+"\n\n"+
        md_table(schema_profile.loc[schema_profile.id.isin(ids)&(
            schema_profile.inferred_dtype.eq("mixed numeric/text")|schema_profile.leading_zero_count.gt(0)|
            schema_profile.encoding_suspect_count.gt(0)|schema_profile.header_encoding_suspect|
            schema_profile.duplicate_header|schema_profile.blank_header),
            ["id","column_position","raw_column","storage_dtype","inferred_dtype","leading_zero_count","encoding_suspect_count","header_encoding_suspect"]]),
        "## 6. Cleaning decision table\n\n"+md_table(local_dec[[
            "issue_id","file","table","field","issue_type","affected_count","count_basis","raw_examples",
            "proposed_rule","canonical_type_unit","handling","evidence","confidence","required_approval","validation_test","origin"]]),
        "## 7. Canonical Silver schema recommendations\n\n"+md_table(canonical_schema(source,representative))+
        "\n\nSchema là đề xuất, chưa cast/convert/write Raw hay Silver. Preserve raw lexical identifiers, measure values/units, source flags/notes, period labels và lineage. "+
        "Decimal precision/scale phải chọn theo source precision và downstream requirements; không tự làm tròn để PASS.",
        "## 8. Business-key candidates\n\n"+md_table(local_keys)+
        "\n\nCandidate ≠ approved unique constraint. duplicate_participants gồm cả bản gốc; duplicate_excess là phần dư; conflict_groups "+
        "xét distinct full-row variants, value_conflict_groups chỉ xét distinct measure values. Không chọn winner theo file order/ingestion timestamp. "+
        "Nếu release metadata thiếu, source_content_sha256 chỉ là snapshot lineage, không đại diện business vintage.",
        "## 9. Proposed DQ / quarantine rules\n\n"+
        md_table(local_dec.loc[local_dec.handling.isin(["QUARANTINE","FLAG","EXCLUDE_STRUCTURAL"]),
            ["issue_id","issue_type","affected_count","count_basis","handling","proposed_rule","validation_test"]])+
        "\n\nQuarantine chỉ áp sau khi grain/parser/reference rules được review. Không quarantine tự động missing tokens, negatives, residuals, "+
        "outliers hoặc nhãn tỉnh lịch sử. Key conflict pending cần xác định grain/release trước; giữ đầy đủ evidence, không last-wins.",
        "## 10. Unknowns và required approvals\n\n"+str(info.get("unknowns","See decisions"))+"\n\n"+
        md_table(local_dec.loc[local_dec.handling.eq("PENDING_REVIEW")|local_dec.confidence.ne("VERIFIED"),
            ["issue_id","issue_type","confidence","handling","required_approval"]])+
        "\n\nReview source-specific period basis, units, geography versions, missing tokens, exact/semantic duplicate policy and aggregation/additivity. "+
        "Approve implementation only after this spec is reviewed; task hiện tại không viết Silver.",
        "## 11. Validation checklist\n\n"+
        "\n".join("- [ ] "+str(row.issue_id)+": "+str(row.validation_test) for row in local_dec.itertuples())+
        "\n- [ ] Raw input hashes unchanged; inventory/file/sheet coverage reconciled.\n"+
        "- [ ] Every retained observation has file/sheet/raw row/raw column/value/unit lineage.\n"+
        "- [ ] Missing candidates and parser failures reconcile; zero not silently NULL.\n"+
        "- [ ] Candidate keys checked again after approved normalization; conflicts handled explicitly.\n"+
        "- [ ] Canonical output keeps source period basis, statistic kind and raw quality flags.\n\n"+
        "Notebook execution invariants for this snapshot:\n\n"+md_table(validation_results.loc[
            ~validation_results["check"].str.startswith("Lineage:")|validation_results["check"].str.contains("|".join(re.escape(x) for x in ids),regex=True)]),
    ]
    headers=[]
    for ident in ids:
        entry=DATA[ident];m=entry["meta"];d=entry["raw"]
        literal=json.dumps(m["headers"],ensure_ascii=False,default=str)
        fence=chr(96)*3
        text=f"### {ident}\n\nRaw records: {len(d)}; columns: {d.shape[1]}; sheet/CSV physical rows: {m['physical_rows']}; header rows: {m['header_rows']}.\n\nLiteral headers/positional grid labels:\n\n"+fence+"text\n"+literal+"\n"+fence
        if ident in WB_CONTEXT:
            text+="\n\nDerived header names + source unit row (projection only):\n\n"+md_table(pd.DataFrame([
                {"raw_column_ordinal":int(c)+1,"series_raw":name,"raw_unit":WB_CONTEXT[ident]["source_units"][c]}
                for c,name in WB_CONTEXT[ident]["series_names"].items()]))
        if m["role"]=="metadata" and len(d):
            if m["table"]=="Description":
                rice_rows=[(int(idx)+1,as_text(row.iloc[1])) for idx,row in d.iterrows() if len(row)>1 and as_text(row.iloc[1]).startswith("Rice (")]
                text+="\n\nRice definitions from source Description sheet:\n\n"+md_table(pd.DataFrame(rice_rows,columns=["physical_row","definition"]))
        headers.append(text)
    sections.append("## 12. Mapping tới file Raw, observed rows/columns và cross-source evidence\n\n"+
        "CSV row = header-inclusive record ordinal (multiline record có thể khác physical line). Excel row = worksheet physical row. "+
        "HTML row = table row ordinal (header row 1), không phải HTML text line. raw_column giữ literal field/year hoặc 1-based ordinal. "+
        "Issue table cung cấp representative exact coordinates; toàn bộ masks/profiles ở notebook để drill down.\n\n"+
        "\n\n".join(headers)+"\n\nRelevant cross-source checks (diagnostic; incompatibilities retained):\n\n"+
        md_table(cross_checks.loc[cross_checks.relation.str.contains(
            "NSO" if source=="nso" else "USDA|Milling|World Bank" if source=="usda" else
            "World Bank" if source=="world_bank" else "Trade|FAOSTAT|price" if source=="faostat" else "domestic",case=False,regex=True)])+
        ("\n\nOfficial evidence for Milling Rate: [Vietnam annual 2010]("+OFFICIAL_EVIDENCE["milling_vietnam"]+
         "), [TU1012 rice formula, printed page 17]("+OFFICIAL_EVIDENCE["milling_formula"]+
         "). Cross-source inference uses these historical documents and same-Raw rough/milled ratios. "+
         "The 1/10000 encoding is supported; per-year period starts and release status remain pending." if source=="usda" else "")+
        "\n\nKhông kế thừa quy tắc từ tài liệu/Silver cũ; mọi proposal có origin trong decision table.")
    return "\n\n".join(sections)+"\n"

SPEC_TEXTS={path:make_spec(path,ids) for path,ids in SPEC_PLAN.items()}
def export_specs():
    created=[]
    for relative,content in SPEC_TEXTS.items():
        base=RAW_ROOT/relative
        target=base
        revision=0
        while target.exists():
            if target.read_text(encoding="utf-8")==content:
                created.append({"file":target.relative_to(RAW_ROOT).as_posix(),"status":"identical; preserved"})
                break
            revision+=1
            suffix="_v2" if revision==1 else "_v2_"+str(revision)
            target=base.with_name(base.stem+suffix+base.suffix)
        else:
            if target.parent.resolve() not in {(RAW_ROOT/s).resolve() for s in SOURCE_NAMES} or not target.name.startswith("cleaning_"):
                raise ValueError("Exporter target outside permitted source specification paths")
            with target.open("x",encoding="utf-8",newline="\n") as stream:
                stream.write(content)
            created.append({"file":target.relative_to(RAW_ROOT).as_posix(),"status":"created"})
    return pd.DataFrame(created)
if WRITE_SPECS:
    assert validation_results.status.eq("PASS").all()
    exported_specs=export_specs()
    show(exported_specs,50)
else:
    print(f"Prepared {len(SPEC_TEXTS)} Markdown specifications in memory; exporter disabled. No data or pipeline writes.")
    display(pd.DataFrame([{"target":p,"tables":len(SPEC_PLAN[p]),"decision_rows":int(decisions.id.isin(SPEC_PLAN[p]).sum()),
        "characters":len(content)} for p,content in SPEC_TEXTS.items()]))


Prepared 10 Markdown specifications in memory; exporter disabled. No data or pipeline writes.


,target,tables,decision_rows,characters
0,faostat/cleaning_monthly_price.md,1,6,16452
1,faostat/cleaning_production.md,1,5,15783
2,faostat/cleaning_supply_utilization.md,1,6,16617
3,faostat/cleaning_trade.md,6,11,27089
4,nso/cleaning_schema_group_01.md,1,5,18322
5,nso/cleaning_schema_group_02.md,12,72,88355
6,thitruongnongsan/cleaning_prices.md,1,8,17697
7,usda/cleaning_rice_yearbook.md,1,4,16353
8,usda/cleaning_psd.md,1,4,16518
9,world_bank/cleaning_pinksheet.md,5,11,43556
